# Retrieval

Ноутбук строит candidate pool для реранкера из нескольких retrieval-источников:
- TopPopular;
- ALS;
- BM25 item-to-item;
- category/user-category popularity;
- CLIP/fCLIP profile retrieval через ANN.

BM25 item-to-item реализован через on-demand sparse retrieval без материализации полной item-item матрицы.

## Imports and paths

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import pickle
import shutil
import warnings
from collections import OrderedDict, defaultdict
from dataclasses import asdict, dataclass
from datetime import date, datetime, timedelta, timezone
from functools import reduce
from pathlib import Path
from typing import Iterable

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import polars as pl
from scipy.sparse import csr_matrix, load_npz, save_npz
from tqdm.auto import tqdm, trange

try:
    from implicit.als import AlternatingLeastSquares
except ImportError as e:
    raise ImportError("Не установлен пакет implicit: pip install implicit") from e

try:
    from implicit.nearest_neighbours import BM25Recommender
    HAS_BM25 = True
except Exception as e:
    BM25Recommender = None
    HAS_BM25 = False
    BM25_IMPORT_ERROR = e

try:
    import hnswlib
    HAS_HNSWLIB = True
except Exception as e:
    hnswlib = None
    HAS_HNSWLIB = False
    HNSWLIB_IMPORT_ERROR = e

try:
    import faiss
    HAS_FAISS = True
except Exception as e:
    faiss = None
    HAS_FAISS = False
    FAISS_IMPORT_ERROR = e


def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists() or (p / "data" / "datamart").exists():
            return p
    raise FileNotFoundError("Не найден project root с data/raw или data/datamart")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DATAMART_DIR = DATA_DIR / "datamart"
PROCESSED_DIR = DATA_DIR / "processed" / "candidate_generators_enhanced"
MODELS_DIR = PROJECT_ROOT / "models" / "candidate_generators_enhanced"
CANDIDATES_DIR = DATAMART_DIR / "candidates"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

for path in [PROCESSED_DIR, MODELS_DIR, CANDIDATES_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATAMART_DIR:", DATAMART_DIR)
print("CANDIDATES_DIR:", CANDIDATES_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("HAS_BM25:", HAS_BM25, "(optional; BM25 retrieval is implemented on demand)")
print("HAS_HNSWLIB:", HAS_HNSWLIB)
print("HAS_FAISS:", HAS_FAISS)
if not HAS_HNSWLIB:
    print("NOTE: clip_hnsw will be skipped unless hnswlib is installed: pip install hnswlib")
if not HAS_FAISS:
    print("NOTE: clip_faiss will be skipped unless faiss is installed: pip install faiss-cpu")


@dataclass(frozen=True)
class EnhancedCandidateConfig:
    # Reranker train split: candidates/features before train_target.
    ranker_train_context_start: date = date(2025, 4, 18)
    ranker_train_context_end: date = date(2025, 6, 3)
    ranker_train_target_start: date = date(2025, 6, 4)
    ranker_train_target_end: date = date(2025, 6, 17)

    # Validation split.
    val_context_start: date = date(2025, 4, 18)
    val_context_end: date = date(2025, 6, 17)
    val_target_start: date = date(2025, 6, 18)
    val_target_end: date = date(2025, 7, 1)

    # Final/test split.
    full_context_start: date = date(2025, 5, 2)
    full_context_end: date = date(2025, 7, 1)
    test_target_start: date = date(2025, 7, 2)
    test_target_end: date = date(2025, 7, 15)

    n_shards: int = 128
    metric_ks: tuple[int, ...] = (20, 50, 100, 200, 700)

   # Source switches.
    run_top_popular: bool = True
    run_als: bool = True
    run_i2i_bm25: bool = True
    run_clip_hnsw: bool = True
    # FAISS is more useful as a challenger/benchmark. Set False if local CPU is slow.
    run_clip_faiss: bool = False
    run_user_category_popular: bool = True

# Per-source K. The merged pool is wider than the baseline retrieval pool.
    top_popular_k: int = 200
    als_k: int = 300
    i2i_bm25_k: int = 300
    clip_hnsw_k: int = 200
    clip_faiss_k: int = 200
    user_category_popular_k: int = 200
    merged_max_candidates_per_user: int = 700

    # TopPopular / active universe.
    top_popular_score_mode: str = "orders_views"  # orders_views | engagement
    top_popular_pool_size: int = 10_000
    # 3 days was very restrictive. 14 days gives better candidate recall while still avoiding dead items.
    active_items_days: int | None = 14
    min_delivered_orders_for_active: int = 0

    # Filtering already known items.
    filter_ordered_items: bool = True
    filter_seen_event_items: bool = False

    # Interaction weights.
    weight_preset: str = "current"  # current | ecup_soft | hybrid
    weak_max_cap: float = 20.0
    time_decay_enabled: bool = True
    time_decay_tau_days: float = 30.0
    time_decay_min_factor: float = 0.10

    event_weight_page_view: float = 1.0
    event_weight_view_description: float = 2.0
    event_weight_review_view: float = 3.0
    event_weight_favorite: float = 20.0
    event_weight_to_cart: float = 100.0
    order_weight: float = 100.0

    # ALS.
    als_factors: int = 128
    als_regularization: float = 0.01
    als_iterations: int = 20
    als_alpha: float = 1.0
    als_use_gpu: bool = False
    als_random_state: int = 42
    als_num_threads: int = 0
    als_batch_size: int = 1024
    als_oversample: int = 1000

    # I2I BM25.
    # item-item graph: score(u, item) = history(u) @ X_item_user @ X_item_user.T.
    i2i_neighbors_per_item: int = 120  # reserved for precomputed I2I retrieval.
    i2i_recent_history_k: int = 20
    i2i_cache_size: int = 200_000  # reserved for cached I2I retrieval.
    i2i_bm25_k1: float = 1.2
    i2i_bm25_b: float = 0.75
    i2i_bm25_batch_size: int = 64
    i2i_bm25_cousers_k: int = 1500
    i2i_bm25_min_couser_score: float = 1e-8
    i2i_bm25_min_candidate_score: float = 1e-8

    # User category popular.
    user_category_top_categories: int = 5
    user_category_items_per_category: int = 120

    # CLIP/fclip user-profile ANN.
    clip_history_k: int = 50
    clip_oversample: int = 400
    normalize_clip_vectors: bool = True

    # HNSW backend for fast CPU approximate candidate generation.
    hnsw_m: int = 32
    hnsw_ef_construction: int = 200
    hnsw_ef_search: int = 150
    hnsw_num_threads: int = -1

    # FAISS backend. `ivf_flat` is scalable; `flat` is exact but can be slow for many users/items.
    faiss_index_type: str = "ivf_flat"  # flat | ivf_flat
    faiss_nlist: int = 4096
    faiss_nprobe: int = 64
    faiss_train_sample_size: int = 200_000
    faiss_use_gpu: bool = False

    # Large data controls.
    csr_n_shards: int = 32
    rebuild_csr: bool = False
    rebuild_models: bool = False
    rebuild_candidate_dirs: bool = False
    # Allows rebuilding BM25 candidates independently from ALS/CLIP candidates.
    rebuild_i2i_bm25_candidates: bool = True
    rebuild_i2i_bm25_model: bool = False
    rebuild_merged_candidates: bool = True
    rebuild_aux: bool = False
    max_rows_in_memory: int = 5_000_000


CFG = EnhancedCandidateConfig()
CFG

PROJECT_ROOT: c:\Users\denis\Documents\Repo\ozon
DATAMART_DIR: c:\Users\denis\Documents\Repo\ozon\data\datamart
CANDIDATES_DIR: c:\Users\denis\Documents\Repo\ozon\data\datamart\candidates
PROCESSED_DIR: c:\Users\denis\Documents\Repo\ozon\data\processed\candidate_generators_enhanced
MODELS_DIR: c:\Users\denis\Documents\Repo\ozon\models\candidate_generators_enhanced
HAS_BM25: True (not required for v8 on-demand BM25)
HAS_HNSWLIB: True
HAS_FAISS: True


EnhancedCandidateConfig(ranker_train_context_start=datetime.date(2025, 4, 18), ranker_train_context_end=datetime.date(2025, 6, 3), ranker_train_target_start=datetime.date(2025, 6, 4), ranker_train_target_end=datetime.date(2025, 6, 17), val_context_start=datetime.date(2025, 4, 18), val_context_end=datetime.date(2025, 6, 17), val_target_start=datetime.date(2025, 6, 18), val_target_end=datetime.date(2025, 7, 1), full_context_start=datetime.date(2025, 5, 2), full_context_end=datetime.date(2025, 7, 1), test_target_start=datetime.date(2025, 7, 2), test_target_end=datetime.date(2025, 7, 15), n_shards=128, metric_ks=(20, 50, 100, 200, 700), run_top_popular=True, run_als=True, run_i2i_bm25=True, run_clip_hnsw=True, run_clip_faiss=False, run_user_category_popular=True, top_popular_k=200, als_k=300, i2i_bm25_k=300, clip_hnsw_k=200, clip_faiss_k=200, user_category_popular_k=200, merged_max_candidates_per_user=700, top_popular_score_mode='orders_views', top_popular_pool_size=10000, active_items_day

## I/O helpers

In [2]:
def path_has_parquet(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


def scan_parquet(path: Path | str, *, hive_partitioning: bool = True) -> pl.LazyFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.is_dir():
        if not path_has_parquet(path):
            raise FileNotFoundError(f"No parquet files found in {path}")
        return pl.scan_parquet(str(path / "**" / "*.parquet"), hive_partitioning=hive_partitioning)
    return pl.scan_parquet(str(path), hive_partitioning=hive_partitioning)


def collect_streaming(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)


def lazy_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        return list(lf.collect_schema().names())
    except Exception:
        return list(lf.schema.keys())


def ensure_numeric_cols(lf: pl.LazyFrame, cols: Iterable[str], dtype=pl.Float32) -> pl.LazyFrame:
    existing = set(lazy_columns(lf))
    exprs = [pl.lit(0).cast(dtype).alias(c) for c in cols if c not in existing]
    return lf.with_columns(exprs) if exprs else lf


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def clean_output_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return str(path)


def events_path(level: str) -> Path:
    return DATAMART_DIR / "agg" / "events" / level


def orders_path(level: str) -> Path:
    return DATAMART_DIR / "agg" / "orders" / level


def _date_filter(lf: pl.LazyFrame, date_col: str, start: date, end: date) -> pl.LazyFrame:
    return (
        lf.with_columns(pl.col(date_col).cast(pl.Date, strict=False).alias(date_col))
        .filter((pl.col(date_col) >= pl.lit(start).cast(pl.Date)) & (pl.col(date_col) <= pl.lit(end).cast(pl.Date)))
    )


def read_events_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    return _date_filter(scan_parquet(events_path(level), hive_partitioning=True), "event_date", start, end)


def read_orders_lf(level: str, start: date, end: date) -> pl.LazyFrame:
    return _date_filter(scan_parquet(orders_path(level), hive_partitioning=True), "created_date", start, end)


def candidate_files(path: Path) -> list[Path]:
    return sorted(path.rglob("*.parquet")) if path.exists() else []


def scan_candidate_shard_or_none(path: Path, shard_id: int) -> pl.LazyFrame | None:
    shard_dir = path / f"user_shard={shard_id}"
    if not shard_dir.exists() or not any(shard_dir.glob("*.parquet")):
        return None
    return pl.scan_parquet(str(shard_dir / "*.parquet"), hive_partitioning=False)


def write_single_shard(part: pl.DataFrame, out_dir: Path, shard_id: int, filename: str) -> int:
    shard_dir = out_dir / f"user_shard={shard_id}"
    shard_dir.mkdir(parents=True, exist_ok=True)
    out_file = shard_dir / filename
    part.write_parquet(out_file, compression="zstd")
    return part.height

## Metrics and evaluation

In [3]:
def _as_unique_list(values) -> list[int]:
    if values is None:
        return []
    if isinstance(values, set):
        return [int(x) for x in values]
    return list(dict.fromkeys(int(x) for x in values))


def hit_rate_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_as_unique_list(gt_items))
    if not gt:
        return np.nan
    return float(any(int(x) in gt for x in pred_items[:k]))


def recall_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_as_unique_list(gt_items))
    if not gt:
        return np.nan
    return float(len(gt & set(int(x) for x in pred_items[:k])) / len(gt))


def ndcg_at_k(gt_items, pred_items, k: int) -> float:
    gt = set(_as_unique_list(gt_items))
    if not gt:
        return np.nan
    dcg = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            dcg += 1.0 / np.log2(rank + 1)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, min(len(gt), k) + 1))
    return float(dcg / idcg) if idcg else 0.0


def ap_at_k(gt_items, pred_items, k: int | None = None) -> float:
    gt = set(_as_unique_list(gt_items))
    if not gt:
        return np.nan
    if k is None:
        k = len(pred_items)
    hits = 0
    score = 0.0
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            hits += 1
            score += hits / rank
    return float(score / len(gt))


def rr_at_k(gt_items, pred_items, k: int | None = None) -> float:
    gt = set(_as_unique_list(gt_items))
    if not gt:
        return np.nan
    if k is None:
        k = len(pred_items)
    for rank, item in enumerate(pred_items[:k], start=1):
        if int(item) in gt:
            return float(1.0 / rank)
    return 0.0


def build_ground_truth(start: date, end: date) -> dict[int, list[int]]:
    df = collect_streaming(
        read_orders_lf("user_item", start, end)
        .filter(pl.col("cnt_orders").fill_null(0) > 0)
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64))
        .unique()
        .sort(["user_id", "item_id"])
    )
    result: dict[int, list[int]] = defaultdict(list)
    for u, i in zip(df["user_id"].to_numpy(), df["item_id"].to_numpy()):
        result[int(u)].append(int(i))
    return dict(result)


def evaluate_candidate_dir(
    candidates_dir: Path,
    ground_truth: dict[int, list[int]],
    *,
    ks: Iterable[int] = CFG.metric_ks,
    rank_col: str = "candidate_rank",
    label: str = "model",
    n_shards: int = CFG.n_shards,
) -> pd.DataFrame:
    ks = tuple(int(k) for k in ks)
    sums = {k: defaultdict(float) for k in ks}
    counts = []
    users_eval = 0
    unique_items: set[int] = set()

    for shard_id in trange(n_shards, desc=f"Evaluate {label}"):
        lf = scan_candidate_shard_or_none(candidates_dir, shard_id)
        if lf is None:
            continue
        part = collect_streaming(
            lf.select(
                pl.col("user_id").cast(pl.Int64),
                pl.col("item_id").cast(pl.Int64),
                pl.col(rank_col).cast(pl.Int32).alias("eval_rank"),
            ).sort(["user_id", "eval_rank", "item_id"])
        )
        if part.is_empty():
            continue
        pdf = part.to_pandas()
        for user_id, group in pdf.groupby("user_id", sort=False):
            user_id = int(user_id)
            gt = ground_truth.get(user_id)
            if not gt:
                continue
            pred = group["item_id"].astype(np.int64).tolist()
            users_eval += 1
            counts.append(len(pred))
            unique_items.update(pred)
            for k in ks:
                sums[k]["hit_rate"] += hit_rate_at_k(gt, pred, k)
                sums[k]["recall"] += recall_at_k(gt, pred, k)
                sums[k]["ndcg"] += ndcg_at_k(gt, pred, k)
                sums[k]["map"] += ap_at_k(gt, pred, k)
                sums[k]["mrr"] += rr_at_k(gt, pred, k)
        del pdf, part
        gc.collect()

    rows = []
    total_gt = len(ground_truth)
    for k in ks:
        row = {
            "model": label,
            "k": k,
            "users_total_gt": total_gt,
            "users_evaluated": users_eval,
            "coverage_users": users_eval / total_gt if total_gt else np.nan,
            "avg_candidates_per_user": float(np.mean(counts)) if counts else 0.0,
            "p50_candidates_per_user": float(np.percentile(counts, 50)) if counts else 0.0,
            "p95_candidates_per_user": float(np.percentile(counts, 95)) if counts else 0.0,
            "users_with_less_than_k": int(sum(c < k for c in counts)),
            "unique_items": len(unique_items),
        }
        for metric in ["hit_rate", "recall", "ndcg", "map", "mrr"]:
            row[metric] = float(sums[k][metric] / users_eval) if users_eval else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def source_contribution_recall_report(
    merged_candidates_dir: Path,
    ground_truth: dict[int, list[int]],
    *,
    ks: Iterable[int] = CFG.metric_ks,
    rank_col: str = "candidate_rank",
    n_shards: int = CFG.n_shards,
) -> pd.DataFrame:
    ks = tuple(int(k) for k in ks)
    total_gt_items = int(sum(len(v) for v in ground_truth.values()))
    source_cols: set[str] = set()
    for f in candidate_files(merged_candidates_dir)[:5]:
        cols = pl.read_parquet(f, n_rows=1).columns
        source_cols.update(c for c in cols if c.startswith("in_"))
    source_cols = sorted(source_cols)
    counters = {k: {"found_union": 0, **{c: 0 for c in source_cols}} for k in ks}

    for shard_id in trange(n_shards, desc="Source contribution"):
        lf = scan_candidate_shard_or_none(merged_candidates_dir, shard_id)
        if lf is None:
            continue
        cols = ["user_id", "item_id", rank_col] + source_cols
        part = collect_streaming(lf.select([pl.col(c) for c in cols if c in lazy_columns(lf)]).sort(["user_id", rank_col, "item_id"]))
        if part.is_empty():
            continue
        pdf = part.to_pandas()
        for user_id, group in pdf.groupby("user_id", sort=False):
            gt = set(ground_truth.get(int(user_id), []))
            if not gt:
                continue
            for k in ks:
                top = group.head(k)
                found = set(int(x) for x in top["item_id"].tolist()) & gt
                counters[k]["found_union"] += len(found)
                if not found:
                    continue
                for c in source_cols:
                    if c in top.columns:
                        sub = top[top[c].fillna(0).astype(int) == 1]
                        counters[k][c] += len(set(int(x) for x in sub["item_id"].tolist()) & gt)
        del pdf, part
        gc.collect()

    rows = []
    for k in ks:
        row = {"k": k, "total_gt_items": total_gt_items}
        for key, val in counters[k].items():
            row[key] = val
            row[f"{key}_recall_micro"] = val / total_gt_items if total_gt_items else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

## Split users and filters

In [4]:
def build_active_items_file(start: date, end: date, active_items_days: int | None, out_dir: Path) -> Path | None:
    if active_items_days is None and CFG.min_delivered_orders_for_active <= 0:
        return None
    ensure_dir(out_dir)
    suffix = f"days={active_items_days}_min_deliv={CFG.min_delivered_orders_for_active}"
    out = out_dir / f"active_items_{suffix}_{end}.parquet"
    if out.exists() and not CFG.rebuild_aux:
        return out

    parts = []
    if active_items_days is not None:
        active_start = end - timedelta(days=active_items_days - 1)
        parts.append(read_events_lf("user_item", active_start, end).select(pl.col("item_id").cast(pl.Int64)))
        parts.append(read_orders_lf("user_item", active_start, end).select(pl.col("item_id").cast(pl.Int64)))

    if CFG.min_delivered_orders_for_active > 0:
        parts.append(
            read_orders_lf("item", start, end)
            .filter(pl.col("cnt_delivered_orders").fill_null(0) >= int(CFG.min_delivered_orders_for_active))
            .select(pl.col("item_id").cast(pl.Int64))
        )

    active_df = collect_streaming(pl.concat(parts).unique().sort("item_id"))
    active_df.write_parquet(out, compression="zstd")
    print(f"active items: {active_df.height:,} -> {rel(out)}")
    return out


def apply_active_items_filter(lf: pl.LazyFrame, active_items_path: Path | None) -> pl.LazyFrame:
    if active_items_path is None:
        return lf
    active_lf = scan_parquet(active_items_path, hive_partitioning=False).select(pl.col("item_id").cast(pl.Int64))
    return lf.with_columns(pl.col("item_id").cast(pl.Int64)).join(active_lf, on="item_id", how="inner")


def build_ordered_by_user(start: date, end: date, *, as_idx: bool = False, user_map_path: Path | None = None, item_map_path: Path | None = None) -> dict[int, set[int]]:
    lf = read_orders_lf("user_item", start, end).select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64)).unique()
    if as_idx:
        if user_map_path is None or item_map_path is None:
            raise ValueError("user_map_path and item_map_path are required")
        lf = (
            lf.join(scan_parquet(user_map_path, hive_partitioning=False).select("user_id", "user_idx"), on="user_id", how="inner")
            .join(scan_parquet(item_map_path, hive_partitioning=False).select("item_id", "item_idx"), on="item_id", how="inner")
            .select(pl.col("user_idx").cast(pl.Int32), pl.col("item_idx").cast(pl.Int32))
        )
        key_col, value_col = "user_idx", "item_idx"
    else:
        key_col, value_col = "user_id", "item_id"
    df = collect_streaming(lf)
    result: dict[int, set[int]] = defaultdict(set)
    for u, i in zip(df[key_col].to_numpy(), df[value_col].to_numpy()):
        result[int(u)].add(int(i))
    return dict(result)


def build_seen_events_by_user(start: date, end: date) -> dict[int, set[int]]:
    df = collect_streaming(read_events_lf("user_item", start, end).select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64)).unique())
    result: dict[int, set[int]] = defaultdict(set)
    for u, i in zip(df["user_id"].to_numpy(), df["item_id"].to_numpy()):
        result[int(u)].add(int(i))
    return dict(result)


def find_raw_test_orders_path() -> Path | None:
    names = [
        "ml_ozon_recsys_test_final_apparel_orders_data",
        "test_orders",
        "orders_test",
    ]
    for name in names:
        p = RAW_DIR / name
        if p.exists():
            return p
    if RAW_DIR.exists():
        matched = sorted(p for p in RAW_DIR.iterdir() if "test" in p.name.lower() and "order" in p.name.lower())
        if matched:
            return matched[0]
    return None


def load_test_users_fallback(start: date, end: date) -> list[int]:
    test_orders_path = find_raw_test_orders_path()
    if test_orders_path is not None:
        print("test orders path:", test_orders_path)
        df = collect_streaming(scan_parquet(test_orders_path, hive_partitioning=True).select(pl.col("user_id").cast(pl.Int64)).drop_nulls().unique().sort("user_id"))
        return [int(x) for x in df["user_id"].to_numpy()]
    print("Raw test orders not found. Fallback: users from datamart orders in test window.")
    return sorted(build_ground_truth(start, end).keys())

## Weights and TopPopular

In [5]:
def time_decay_expr(date_col: str, ref_date: date) -> pl.Expr:
    if not CFG.time_decay_enabled:
        return pl.lit(1.0, dtype=pl.Float32)
    age = (pl.lit(ref_date).cast(pl.Date) - pl.col(date_col).cast(pl.Date)).dt.total_days().cast(pl.Float32)
    age = pl.when(age < 0).then(0.0).otherwise(age)
    decay = (-age / float(CFG.time_decay_tau_days)).exp()
    decay = pl.when(decay < float(CFG.time_decay_min_factor)).then(float(CFG.time_decay_min_factor)).otherwise(decay)
    return decay.cast(pl.Float32)


def weight_values() -> dict[str, float]:
    if CFG.weight_preset == "current":
        return dict(page_view=CFG.event_weight_page_view, view_description=CFG.event_weight_view_description, review_view=CFG.event_weight_review_view, favorite=CFG.event_weight_favorite, to_cart=CFG.event_weight_to_cart, order=CFG.order_weight)
    if CFG.weight_preset == "ecup_soft":
        return dict(page_view=2.0, view_description=3.0, review_view=3.0, favorite=5.0, to_cart=4.0, order=5.0)
    if CFG.weight_preset == "hybrid":
        return dict(page_view=1.0, view_description=2.0, review_view=3.0, favorite=10.0, to_cart=30.0, order=50.0)
    raise ValueError(CFG.weight_preset)


def _read_item_popularity_lf(domain: str, start: date, end: date) -> pl.LazyFrame:
    if domain == "events":
        event_cols = ["cnt_events", "cnt_page_view", "cnt_view_description", "cnt_review_view", "cnt_to_cart", "cnt_favorite"]
        if path_has_parquet(events_path("item")):
            return ensure_numeric_cols(read_events_lf("item", start, end), event_cols)
        lf = ensure_numeric_cols(read_events_lf("user_item", start, end), event_cols)
        return lf.group_by(["event_date", "item_id"]).agg(*[pl.sum(c).alias(c) for c in event_cols])
    if domain == "orders":
        order_cols = ["cnt_orders", "cnt_delivered_orders", "cnt_canceled_orders", "cnt_proccesed_orders"]
        if path_has_parquet(orders_path("item")):
            return ensure_numeric_cols(read_orders_lf("item", start, end), order_cols)
        lf = ensure_numeric_cols(read_orders_lf("user_item", start, end), order_cols)
        return lf.group_by(["created_date", "item_id"]).agg(*[pl.sum(c).alias(c) for c in order_cols])
    raise ValueError(domain)


def build_top_popular_table(start: date, end: date, *, out_dir: Path, label: str) -> pd.DataFrame:
    active_items_path = build_active_items_file(start, end, CFG.active_items_days, out_dir)
    events_lf = apply_active_items_filter(_read_item_popularity_lf("events", start, end), active_items_path)
    orders_lf = apply_active_items_filter(_read_item_popularity_lf("orders", start, end), active_items_path)

    events_df = collect_streaming(
        events_lf.group_by("item_id").agg(
            pl.sum("cnt_events").alias("event_cnt"),
            pl.sum("cnt_page_view").alias("page_view_cnt"),
            pl.sum("cnt_view_description").alias("view_description_cnt"),
            pl.sum("cnt_review_view").alias("review_view_cnt"),
            pl.sum("cnt_to_cart").alias("to_cart_cnt"),
            pl.sum("cnt_favorite").alias("favorite_cnt"),
        )
    ).to_pandas()
    orders_df = collect_streaming(
        orders_lf.group_by("item_id").agg(
            pl.sum("cnt_orders").alias("order_cnt"),
            pl.sum("cnt_delivered_orders").alias("delivered_order_cnt"),
            pl.sum("cnt_canceled_orders").alias("canceled_order_cnt"),
            pl.sum("cnt_proccesed_orders").alias("processed_order_cnt"),
        )
    ).to_pandas()

    pop = orders_df.merge(events_df, on="item_id", how="outer").fillna(0)
    for c in [c for c in pop.columns if c != "item_id"]:
        pop[c] = pop[c].astype(np.float32)
    pop["score_orders_views"] = (3.0 * pop["order_cnt"] + pop["page_view_cnt"]).astype(np.float32)
    pop["score_engagement"] = (3.0 * pop["order_cnt"] + pop["page_view_cnt"] + 2.0 * pop["view_description_cnt"] + 3.0 * pop["review_view_cnt"] + 20.0 * pop["favorite_cnt"] + 100.0 * pop["to_cart_cnt"]).astype(np.float32)
    if CFG.top_popular_score_mode == "orders_views":
        pop["score"] = pop["score_orders_views"].astype(np.float32)
    elif CFG.top_popular_score_mode == "engagement":
        pop["score"] = pop["score_engagement"].astype(np.float32)
    else:
        raise ValueError(CFG.top_popular_score_mode)
    pop = pop[pop["score"] > 0].sort_values(["score", "order_cnt", "page_view_cnt", "item_id"], ascending=[False, False, False, True]).reset_index(drop=True)
    pop["global_rank"] = np.arange(1, len(pop) + 1, dtype=np.int32)
    pop["item_id"] = pop["item_id"].astype(np.int64)
    out = out_dir / f"top_popular_table_{label}.parquet"
    ensure_dir(out.parent)
    pop.to_parquet(out, index=False)
    print(f"TopPopular table {label}: {len(pop):,} items -> {rel(out)}")
    return pop


def top_popular_part_for_users(user_ids: list[int], pop_table: pd.DataFrame, *, ordered_by_user: dict[int, set[int]] | None, seen_events_by_user: dict[int, set[int]] | None, k: int, pool_size: int) -> pl.DataFrame:
    pool = pop_table.head(pool_size)[["item_id", "score", "global_rank"]]
    items = pool["item_id"].astype(np.int64).to_numpy()
    scores = pool["score"].astype(np.float32).to_numpy()
    ranks = pool["global_rank"].astype(np.int32).to_numpy()
    ordered_by_user = ordered_by_user or {}
    seen_events_by_user = seen_events_by_user or {}
    up, ip, sp, rp, gp = [], [], [], [], []
    for user_id in user_ids:
        user_id = int(user_id)
        banned = set()
        if CFG.filter_ordered_items:
            banned.update(ordered_by_user.get(user_id, set()))
        if CFG.filter_seen_event_items:
            banned.update(seen_events_by_user.get(user_id, set()))
        if banned:
            selected = np.flatnonzero(~np.isin(items, np.fromiter(banned, dtype=np.int64)))[:k]
        else:
            selected = np.arange(min(k, len(items)), dtype=np.int64)
        if len(selected) == 0:
            continue
        n = len(selected)
        up.append(np.full(n, user_id, dtype=np.int64)); ip.append(items[selected]); sp.append(scores[selected]); rp.append(np.arange(1, n + 1, dtype=np.int32)); gp.append(ranks[selected])
    if not up:
        return pl.DataFrame(schema={"user_id": pl.Int64, "item_id": pl.Int64, "source": pl.Utf8, "score": pl.Float32, "rank": pl.Int32, "global_pop_rank": pl.Int32, "in_top_popular": pl.Int8})
    return pl.DataFrame({"user_id": np.concatenate(up), "item_id": np.concatenate(ip), "score": np.concatenate(sp), "rank": np.concatenate(rp), "global_pop_rank": np.concatenate(gp)}).with_columns(pl.lit("top_popular").alias("source"), pl.lit(1, dtype=pl.Int8).alias("in_top_popular"))


def generate_top_popular_candidates_to_dir(user_ids: list[int], pop_table: pd.DataFrame, out_dir: Path, *, ordered_by_user: dict[int, set[int]], seen_events_by_user: dict[int, set[int]], k: int, n_shards: int = CFG.n_shards) -> Path:
    clean_output_dir(out_dir)
    users_by_shard: dict[int, list[int]] = defaultdict(list)
    for user_id in user_ids:
        users_by_shard[int(user_id) % n_shards].append(int(user_id))
    rows = 0
    for shard_id in trange(n_shards, desc="TopPopular shards"):
        part_users = users_by_shard.get(shard_id, [])
        if not part_users:
            continue
        part = top_popular_part_for_users(part_users, pop_table, ordered_by_user=ordered_by_user, seen_events_by_user=seen_events_by_user, k=k, pool_size=CFG.top_popular_pool_size)
        if not part.is_empty():
            rows += write_single_shard(part, out_dir, shard_id, "top_popular.parquet")
        del part
        gc.collect()
    print(f"TopPopular candidates saved: rows={rows:,} -> {rel(out_dir)}")
    return out_dir

## Interaction CSR and ALS

In [6]:
def build_id_mappings_from_datamart(start: date, end: date, out_dir: Path) -> tuple[Path, Path]:
    ensure_dir(out_dir)
    user_map_path = out_dir / "user_mapping.parquet"
    item_map_path = out_dir / "item_mapping.parquet"
    if user_map_path.exists() and item_map_path.exists() and not CFG.rebuild_csr:
        return user_map_path, item_map_path
    active_items_path = build_active_items_file(start, end, CFG.active_items_days, out_dir)
    events_ui = apply_active_items_filter(read_events_lf("user_item", start, end).select("user_id", "item_id"), active_items_path)
    orders_ui = apply_active_items_filter(read_orders_lf("user_item", start, end).select("user_id", "item_id"), active_items_path)
    user_df = collect_streaming(pl.concat([events_ui.select(pl.col("user_id").cast(pl.Int64)), orders_ui.select(pl.col("user_id").cast(pl.Int64))]).unique().sort("user_id").with_row_index("user_idx").select("user_id", pl.col("user_idx").cast(pl.Int32)))
    item_df = collect_streaming(pl.concat([events_ui.select(pl.col("item_id").cast(pl.Int64)), orders_ui.select(pl.col("item_id").cast(pl.Int64))]).unique().sort("item_id").with_row_index("item_idx").select("item_id", pl.col("item_idx").cast(pl.Int32)))
    user_df.write_parquet(user_map_path, compression="zstd")
    item_df.write_parquet(item_map_path, compression="zstd")
    print(f"users: {user_df.height:,} -> {rel(user_map_path)}")
    print(f"items: {item_df.height:,} -> {rel(item_map_path)}")
    return user_map_path, item_map_path


def build_interactions_lf_for_shard(shard_id: int, n_shards: int, start: date, end: date, active_items_path: Path | None) -> pl.LazyFrame:
    w = weight_values()
    event_cols = ["cnt_page_view", "cnt_view_description", "cnt_review_view", "cnt_to_cart", "cnt_favorite"]
    events_lf = ensure_numeric_cols(read_events_lf("user_item", start, end), event_cols)
    events_lf = apply_active_items_filter(events_lf, active_items_path).filter((pl.col("user_id") % n_shards) == shard_id)
    e_decay = time_decay_expr("event_date", end)
    weak_raw = (pl.col("cnt_page_view").cast(pl.Float32) * w["page_view"] + pl.col("cnt_view_description").cast(pl.Float32) * w["view_description"] + pl.col("cnt_review_view").cast(pl.Float32) * w["review_view"]) * e_decay
    strong_raw = pl.max_horizontal((pl.col("cnt_favorite") > 0).cast(pl.Float32) * w["favorite"], (pl.col("cnt_to_cart") > 0).cast(pl.Float32) * w["to_cart"]) * e_decay
    event_weights = (
        events_lf.with_columns(weak_raw.alias("weak_weight_raw"), strong_raw.alias("strong_weight_raw"))
        .group_by(["user_id", "item_id"])
        .agg(pl.sum("weak_weight_raw").alias("weak_weight_raw"), pl.max("strong_weight_raw").alias("strong_weight"))
        .with_columns(pl.when(pl.col("weak_weight_raw") > CFG.weak_max_cap).then(CFG.weak_max_cap).otherwise(pl.col("weak_weight_raw")).alias("weak_weight"))
        .with_columns(pl.max_horizontal(pl.col("weak_weight"), pl.col("strong_weight")).alias("weight"))
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), pl.col("weight").cast(pl.Float32))
    )
    orders_lf = ensure_numeric_cols(read_orders_lf("user_item", start, end), ["cnt_orders"])
    orders_lf = apply_active_items_filter(orders_lf, active_items_path).filter((pl.col("user_id") % n_shards) == shard_id)
    o_decay = time_decay_expr("created_date", end)
    order_weights = (
        orders_lf.with_columns(((pl.col("cnt_orders") > 0).cast(pl.Float32) * w["order"] * o_decay).alias("weight"))
        .group_by(["user_id", "item_id"]).agg(pl.max("weight").alias("weight"))
        .select(pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64), pl.col("weight").cast(pl.Float32))
    )
    return pl.concat([event_weights, order_weights]).group_by(["user_id", "item_id"]).agg(pl.max("weight").alias("weight")).filter(pl.col("weight") > 0)


def build_csr_from_datamart(start: date, end: date, out_dir: Path, *, n_shards: int = CFG.csr_n_shards) -> tuple[csr_matrix, Path, Path]:
    ensure_dir(out_dir)
    csr_path = out_dir / "interaction_matrix.npz"
    user_map_path, item_map_path = build_id_mappings_from_datamart(start, end, out_dir)
    if csr_path.exists() and not CFG.rebuild_csr:
        print(f"CSR exists: {rel(csr_path)}")
        return load_npz(csr_path), user_map_path, item_map_path
    for old in out_dir.glob("shard_*.npz"):
        old.unlink()
    user_map = pd.read_parquet(user_map_path)
    item_map = pd.read_parquet(item_map_path)
    user2idx = dict(zip(user_map["user_id"].astype(int), user_map["user_idx"].astype(int)))
    item2idx = dict(zip(item_map["item_id"].astype(int), item_map["item_idx"].astype(int)))
    shape = (len(user_map), len(item_map))
    active_items_path = build_active_items_file(start, end, CFG.active_items_days, out_dir)
    for shard_id in trange(n_shards, desc="Build CSR shards"):
        shard_path = out_dir / f"shard_{shard_id}.npz"
        if shard_path.exists() and not CFG.rebuild_csr:
            continue
        inter = collect_streaming(build_interactions_lf_for_shard(shard_id, n_shards, start, end, active_items_path)).to_pandas()
        if inter.empty:
            continue
        inter["u"] = inter["user_id"].map(user2idx)
        inter["i"] = inter["item_id"].map(item2idx)
        inter = inter.dropna(subset=["u", "i"])
        mat = csr_matrix((inter["weight"].astype(np.float32).values, (inter["u"].astype(np.int32).values, inter["i"].astype(np.int32).values)), shape=shape, dtype=np.float32)
        save_npz(shard_path, mat)
        del inter, mat
        gc.collect()
    mats = [load_npz(out_dir / f"shard_{i}.npz") for i in range(n_shards) if (out_dir / f"shard_{i}.npz").exists()]
    if not mats:
        raise RuntimeError(f"No CSR shards were built in {out_dir}")
    matrix = reduce(lambda a, b: a + b, mats)
    save_npz(csr_path, matrix)
    print(f"CSR: shape={matrix.shape}, nnz={matrix.nnz:,} -> {rel(csr_path)}")
    return matrix, user_map_path, item_map_path


def make_als_model():
    kwargs = dict(factors=CFG.als_factors, regularization=CFG.als_regularization, iterations=CFG.als_iterations, alpha=CFG.als_alpha, use_gpu=CFG.als_use_gpu, random_state=CFG.als_random_state)
    if not CFG.als_use_gpu:
        kwargs["num_threads"] = CFG.als_num_threads
    try:
        return AlternatingLeastSquares(**kwargs)
    except TypeError:
        kwargs.pop("num_threads", None)
        return AlternatingLeastSquares(**kwargs)


def train_or_load_als(matrix: csr_matrix, model_path: Path):
    if model_path.exists() and not CFG.rebuild_models:
        proto = make_als_model(); model_cls = type(proto); del proto
        if hasattr(model_cls, "load"):
            try:
                model = model_cls.load(str(model_path))
                print(f"ALS loaded: {rel(model_path)}")
                return model
            except Exception:
                pass
    model = make_als_model()
    model.fit(matrix)
    ensure_dir(model_path.parent)
    if hasattr(model, "save"):
        model.save(str(model_path))
    print(f"ALS saved: {rel(model_path)}")
    return model


def mapping_arrays(user_map_path: Path, item_map_path: Path) -> tuple[np.ndarray, np.ndarray]:
    user_map = pd.read_parquet(user_map_path).sort_values("user_idx")
    item_map = pd.read_parquet(item_map_path).sort_values("item_idx")
    return user_map["user_id"].astype(np.int64).to_numpy(), item_map["item_id"].astype(np.int64).to_numpy()


def make_user_idx_list(user_ids: list[int], user_map_path: Path) -> list[int]:
    user_map = pd.read_parquet(user_map_path)
    user2idx = dict(zip(user_map["user_id"].astype(int), user_map["user_idx"].astype(int)))
    return [int(user2idx[u]) for u in user_ids if int(u) in user2idx]


def als_candidate_part_for_user_idxs(model, matrix: csr_matrix, user_idxs: list[int], user_ids_by_idx: np.ndarray, item_ids_by_idx: np.ndarray, ordered_by_user_idx: dict[int, set[int]], *, k: int, oversample: int, batch_size: int) -> pl.DataFrame:
    n_items = matrix.shape[1]
    up, ip, sp, rp = [], [], [], []
    for start in range(0, len(user_idxs), batch_size):
        batch_users = np.asarray(user_idxs[start:start + batch_size], dtype=np.int32)
        if len(batch_users) == 0:
            continue
        recs, scores = model.recommend(batch_users, matrix[batch_users], N=max(k, oversample), filter_already_liked_items=False, recalculate_user=False)
        recs = np.asarray(recs); scores = np.asarray(scores)
        if recs.ndim == 1:
            recs = recs.reshape(1, -1); scores = scores.reshape(1, -1)
        for u_idx, r, s in zip(batch_users, recs, scores):
            mask = (r >= 0) & (r < n_items)
            if CFG.filter_ordered_items:
                banned = ordered_by_user_idx.get(int(u_idx), set())
                if banned:
                    mask &= ~np.isin(r.astype(np.int32, copy=False), np.fromiter(banned, dtype=np.int32))
            selected = r[mask][:k].astype(np.int64, copy=False)
            selected_scores = s[mask][:k].astype(np.float32, copy=False)
            if len(selected) == 0:
                continue
            n = len(selected)
            up.append(np.full(n, user_ids_by_idx[int(u_idx)], dtype=np.int64)); ip.append(item_ids_by_idx[selected]); sp.append(selected_scores); rp.append(np.arange(1, n + 1, dtype=np.int32))
    if not up:
        return pl.DataFrame(schema={"user_id": pl.Int64, "item_id": pl.Int64, "source": pl.Utf8, "score": pl.Float32, "rank": pl.Int32, "in_als": pl.Int8})
    return pl.DataFrame({"user_id": np.concatenate(up), "item_id": np.concatenate(ip), "score": np.concatenate(sp), "rank": np.concatenate(rp)}).with_columns(pl.lit("als").alias("source"), pl.lit(1, dtype=pl.Int8).alias("in_als"))


def generate_als_candidates_to_dir(model, matrix: csr_matrix, user_idxs: list[int], user_map_path: Path, item_map_path: Path, ordered_by_user_idx: dict[int, set[int]], out_dir: Path, *, k: int, n_shards: int = CFG.n_shards) -> Path:
    clean_output_dir(out_dir)
    user_ids_by_idx, item_ids_by_idx = mapping_arrays(user_map_path, item_map_path)
    by_shard: dict[int, list[int]] = defaultdict(list)
    for u_idx in user_idxs:
        by_shard[int(user_ids_by_idx[int(u_idx)]) % n_shards].append(int(u_idx))
    rows = 0
    for shard_id in trange(n_shards, desc="ALS candidate shards"):
        idxs = by_shard.get(shard_id, [])
        if not idxs:
            continue
        part = als_candidate_part_for_user_idxs(model, matrix, idxs, user_ids_by_idx, item_ids_by_idx, ordered_by_user_idx, k=k, oversample=CFG.als_oversample, batch_size=CFG.als_batch_size)
        if not part.is_empty():
            rows += write_single_shard(part, out_dir, shard_id, "als.parquet")
        del part
        gc.collect()
    print(f"ALS candidates saved: rows={rows:,} -> {rel(out_dir)}")
    return out_dir

## I2I BM25 candidates

In [ ]:
class Bm25OnDemandRetriever:
    """On-demand BM25 I2I retriever.

    The heavy object is only a normalized item-user sparse matrix. We do not
    precompute item-item neighbors. For a batch of users:

        H = sparse user-history matrix, rows = users, cols = items
        X = BM25-weighted normalized item-user matrix, rows = items, cols = users
        C = H @ X       # co-user profile
        S = C @ X.T     # candidate item scores

    This is equivalent to aggregating item-item BM25/cosine similarities over a
    user's recent history, but it avoids the Windows Cython dtype issue in
    implicit and avoids a full item x item materialization.
    """

    def __init__(self, item_user: csr_matrix):
        self.item_user = item_user.tocsr().astype(np.float32, copy=False)
        self.item_user_T = self.item_user.T.tocsr().astype(np.float32, copy=False)


def bm25_weight_user_item_scipy(matrix: csr_matrix, *, k1: float, b: float) -> csr_matrix:
    """BM25-weight a user-item CSR matrix.

    Rows are users/documents, columns are items/terms. The output remains
    user x item and is suitable for item-item/user-profile cosine after transpose.
    """
    X = matrix.tocsr(copy=True).astype(np.float32)
    X.eliminate_zeros()
    X.sort_indices()

    n_users, _ = X.shape
    if X.nnz == 0:
        return X

    row_sums = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
    nonzero_rows = row_sums[row_sums > 0]
    avgdl = float(nonzero_rows.mean()) if len(nonzero_rows) else 1.0
    avgdl = max(avgdl, 1e-6)

    df = np.diff(X.tocsc().indptr).astype(np.float32)
    idf = (np.log(float(max(n_users, 1))) - np.log1p(df)).astype(np.float32)
    idf = np.maximum(idf, 0.0)

    length_norm = ((1.0 - float(b)) + float(b) * row_sums / avgdl).astype(np.float32, copy=False)
    for row in trange(n_users, desc="BM25 weight rows", leave=False):
        lo, hi = X.indptr[row], X.indptr[row + 1]
        if hi <= lo:
            continue
        denom = X.data[lo:hi] + float(k1) * float(length_norm[row])
        numer = X.data[lo:hi] * (float(k1) + 1.0)
        X.data[lo:hi] = (numer / np.maximum(denom, 1e-6) * idf[X.indices[lo:hi]]).astype(np.float32, copy=False)
    X.eliminate_zeros()
    return X


def normalize_csr_rows_l2(matrix: csr_matrix) -> csr_matrix:
    X = matrix.tocsr(copy=False).astype(np.float32, copy=False)
    row_norm = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel()).astype(np.float32)
    inv = np.zeros_like(row_norm, dtype=np.float32)
    mask = row_norm > 0
    inv[mask] = 1.0 / row_norm[mask]
    X = X.multiply(inv[:, None]).tocsr()
    X.eliminate_zeros()
    X.sort_indices()
    return X.astype(np.float32, copy=False)


def prune_csr_rows_topk(matrix: csr_matrix, *, k: int, min_value: float = -np.inf) -> csr_matrix:
    """Keep only top-k values in every CSR row."""
    X = matrix.tocsr(copy=False)
    k = int(k)
    if k <= 0 or X.nnz == 0:
        return csr_matrix(X.shape, dtype=np.float32)

    data_parts, ind_parts = [], []
    indptr = np.zeros(X.shape[0] + 1, dtype=np.int64)
    nnz = 0
    for row in range(X.shape[0]):
        lo, hi = X.indptr[row], X.indptr[row + 1]
        vals = X.data[lo:hi]
        cols = X.indices[lo:hi]
        if len(vals) == 0:
            indptr[row + 1] = nnz
            continue
        mask = np.isfinite(vals) & (vals > float(min_value))
        if not np.any(mask):
            indptr[row + 1] = nnz
            continue
        vals = vals[mask]
        cols = cols[mask]
        kk = min(k, len(vals))
        if kk < len(vals):
            top = np.argpartition(-vals, kk - 1)[:kk]
            top = top[np.argsort(-vals[top])]
            vals = vals[top]
            cols = cols[top]
        else:
            order = np.argsort(-vals)
            vals = vals[order]
            cols = cols[order]
        data_parts.append(vals.astype(np.float32, copy=False))
        ind_parts.append(cols.astype(np.int32, copy=False))
        nnz += len(vals)
        indptr[row + 1] = nnz

    if nnz == 0:
        return csr_matrix(X.shape, dtype=np.float32)
    data = np.concatenate(data_parts).astype(np.float32, copy=False)
    indices = np.concatenate(ind_parts).astype(np.int32, copy=False)
    out = csr_matrix((data, indices, indptr), shape=X.shape, dtype=np.float32)
    out.sort_indices()
    return out


def build_recent_history_matrix(matrix: csr_matrix, batch_user_idxs: np.ndarray, *, history_k: int) -> csr_matrix:
    """Return a batch user-item matrix with only top weighted history items per user."""
    H = matrix[batch_user_idxs].tocsr().astype(np.float32, copy=False)
    H = prune_csr_rows_topk(H, k=int(history_k), min_value=0.0)
    H.eliminate_zeros()
    return H


def train_or_load_i2i_bm25(matrix: csr_matrix, model_path: Path):
    """Build/load the BM25 item-user sparse matrix used by on-demand retrieval."""
    item_user_path = model_path.with_name(model_path.stem.replace("scipy", "ondemand") + "_item_user.npz")
    if item_user_path.exists() and not CFG.rebuild_models and not CFG.rebuild_i2i_bm25_model:
        item_user = load_npz(item_user_path).tocsr().astype(np.float32, copy=False)
        print(f"I2I BM25 on-demand matrix loaded: {rel(item_user_path)} shape={item_user.shape}, nnz={item_user.nnz:,}")
        return Bm25OnDemandRetriever(item_user)

    print("I2I BM25 backend: on-demand sparse BM25 retrieval")
    print(f"input matrix: users={matrix.shape[0]:,}, items={matrix.shape[1]:,}, nnz={matrix.nnz:,}")
    weighted_user_item = bm25_weight_user_item_scipy(matrix, k1=CFG.i2i_bm25_k1, b=CFG.i2i_bm25_b)
    item_user = normalize_csr_rows_l2(weighted_user_item.T.tocsr())
    del weighted_user_item
    gc.collect()

    ensure_dir(item_user_path.parent)
    save_npz(item_user_path, item_user)
    print(f"I2I BM25 on-demand matrix saved: {rel(item_user_path)} shape={item_user.shape}, nnz={item_user.nnz:,}")
    return Bm25OnDemandRetriever(item_user)


def i2i_bm25_candidate_part_for_user_idxs(retriever: Bm25OnDemandRetriever, matrix: csr_matrix, user_idxs: list[int], user_ids_by_idx: np.ndarray, item_ids_by_idx: np.ndarray, ordered_by_user_idx: dict[int, set[int]], *, k: int) -> pl.DataFrame:
    up, ip, sp, rp = [], [], [], []
    n_items = matrix.shape[1]
    batch_size = int(max(1, CFG.i2i_bm25_batch_size))

    for start in tqdm(range(0, len(user_idxs), batch_size), leave=False, desc="I2I BM25 user batches"):
        batch = np.asarray(user_idxs[start:start + batch_size], dtype=np.int32)
        if len(batch) == 0:
            continue

        H = build_recent_history_matrix(matrix, batch, history_k=CFG.i2i_recent_history_k)
        if H.nnz == 0:
            continue

        # users x users co-profile, aggressively pruned before scoring items.
        co_users = (H @ retriever.item_user).tocsr().astype(np.float32, copy=False)
        co_users = prune_csr_rows_topk(co_users, k=CFG.i2i_bm25_cousers_k, min_value=CFG.i2i_bm25_min_couser_score)
        if co_users.nnz == 0:
            continue

        scores_mat = (co_users @ retriever.item_user_T).tocsr().astype(np.float32, copy=False)
        del co_users

        for local_i, u_idx in enumerate(batch):
            lo, hi = scores_mat.indptr[local_i], scores_mat.indptr[local_i + 1]
            cols = scores_mat.indices[lo:hi]
            vals = scores_mat.data[lo:hi]
            if len(vals) == 0:
                continue
            mask = (cols >= 0) & (cols < n_items) & np.isfinite(vals) & (vals > CFG.i2i_bm25_min_candidate_score)
            if not np.any(mask):
                continue
            cols = cols[mask].astype(np.int32, copy=False)
            vals = vals[mask].astype(np.float32, copy=False)

            # Remove items already in user's context history and optionally already ordered.
            hist_lo, hist_hi = H.indptr[local_i], H.indptr[local_i + 1]
            banned_arr = H.indices[hist_lo:hist_hi].astype(np.int32, copy=False)
            if CFG.filter_ordered_items:
                banned = ordered_by_user_idx.get(int(u_idx), set())
                if banned:
                    banned_arr = np.unique(np.concatenate([banned_arr, np.fromiter(banned, dtype=np.int32)]))
            if len(banned_arr):
                keep = ~np.isin(cols, banned_arr)
                cols = cols[keep]
                vals = vals[keep]
            if len(vals) == 0:
                continue

            kk = min(int(k), len(vals))
            if kk < len(vals):
                top = np.argpartition(-vals, kk - 1)[:kk]
                top = top[np.argsort(-vals[top])]
            else:
                top = np.argsort(-vals)
            selected = cols[top]
            selected_scores = vals[top]
            n = len(selected)
            if n == 0:
                continue
            up.append(np.full(n, user_ids_by_idx[int(u_idx)], dtype=np.int64))
            ip.append(item_ids_by_idx[selected.astype(np.int64)])
            sp.append(selected_scores.astype(np.float32, copy=False))
            rp.append(np.arange(1, n + 1, dtype=np.int32))

        del H, scores_mat
        gc.collect()

    if not up:
        return pl.DataFrame(schema={"user_id": pl.Int64, "item_id": pl.Int64, "source": pl.Utf8, "score": pl.Float32, "rank": pl.Int32, "in_i2i_bm25": pl.Int8})
    return pl.DataFrame({"user_id": np.concatenate(up), "item_id": np.concatenate(ip), "score": np.concatenate(sp), "rank": np.concatenate(rp)}).with_columns(pl.lit("i2i_bm25").alias("source"), pl.lit(1, dtype=pl.Int8).alias("in_i2i_bm25"))


def generate_i2i_bm25_candidates_to_dir(retriever: Bm25OnDemandRetriever, matrix: csr_matrix, user_idxs: list[int], user_map_path: Path, item_map_path: Path, ordered_by_user_idx: dict[int, set[int]], out_dir: Path, *, k: int, n_shards: int = CFG.n_shards) -> Path:
    clean_output_dir(out_dir)
    user_ids_by_idx, item_ids_by_idx = mapping_arrays(user_map_path, item_map_path)
    by_shard: dict[int, list[int]] = defaultdict(list)
    for u_idx in user_idxs:
        by_shard[int(user_ids_by_idx[int(u_idx)]) % n_shards].append(int(u_idx))
    rows = 0
    for shard_id in trange(n_shards, desc="I2I BM25 candidate shards"):
        idxs = by_shard.get(shard_id, [])
        if not idxs:
            continue
        part = i2i_bm25_candidate_part_for_user_idxs(retriever, matrix, idxs, user_ids_by_idx, item_ids_by_idx, ordered_by_user_idx, k=k)
        if not part.is_empty():
            rows += write_single_shard(part, out_dir, shard_id, "i2i_bm25.parquet")
        del part
        gc.collect()
    print(f"I2I BM25 candidates saved: rows={rows:,} -> {rel(out_dir)}")
    return out_dir


## Catalog and user-category candidates

In [8]:
def item_catalog_lf_enriched() -> pl.LazyFrame:
    base_path = DATAMART_DIR / "mappings" / "item_catalog.parquet"
    if base_path.exists():
        lf = scan_parquet(base_path, hive_partitioning=False).select(pl.col("item_id").cast(pl.Int64), pl.col("catalogid").cast(pl.Int64)).unique()
    else:
        lf = scan_parquet(events_path("user_item"), hive_partitioning=True).select(pl.col("item_id").cast(pl.Int64), pl.col("catalogid").cast(pl.Int64)).drop_nulls().unique()
    cat_l0_path = MODELS_DIR / "catalogid_to_cat_l0.parquet"
    if cat_l0_path.exists() or build_catalog_l0_mapping(cat_l0_path):
        lf = lf.join(scan_parquet(cat_l0_path, hive_partitioning=False), on="catalogid", how="left")
    else:
        lf = lf.with_columns(pl.col("catalogid").alias("cat_l0"))
    return lf.with_columns(pl.col("cat_l0").fill_null(pl.col("catalogid")).cast(pl.Int64))


def build_catalog_l0_mapping(out_path: Path) -> bool:
    if out_path.exists() and not CFG.rebuild_aux:
        return True
    candidates = [RAW_DIR / "ml_ozon_recsys_train_final_categories_tree", DATA_DIR / "ml_ozon_recsys_train_final_categories_tree"]
    cat_path = next((p for p in candidates if p.exists()), None)
    if cat_path is None:
        return False
    try:
        df = collect_streaming(scan_parquet(cat_path, hive_partitioning=False).select(pl.col("catalogid").cast(pl.Int64), pl.col("ids"))).to_pandas()
        cat_l0 = []
        for ids in df["ids"].tolist():
            if ids is None or len(ids) == 0:
                cat_l0.append(-1)
                continue
            vals = [int(x) for x in ids if int(x) != -1]
            cat_l0.append(vals[-1] if vals else int(ids[0]))
        out = pl.DataFrame({"catalogid": df["catalogid"].astype(np.int64).to_numpy(), "cat_l0": np.asarray(cat_l0, dtype=np.int64)}).unique()
        ensure_dir(out_path.parent)
        out.write_parquet(out_path, compression="zstd")
        print(f"catalogid -> cat_l0: {out.height:,} -> {rel(out_path)}")
        return True
    except Exception as e:
        warnings.warn(f"Cannot build catalog_l0 mapping: {e}")
        return False


def build_user_category_preferences(start: date, end: date, out_dir: Path, label: str) -> Path:
    ensure_dir(out_dir)
    out = out_dir / f"user_category_preferences_{label}.parquet"
    if out.exists() and not CFG.rebuild_aux:
        return out
    w = weight_values()
    event_cols = ["cnt_page_view", "cnt_view_description", "cnt_review_view", "cnt_to_cart", "cnt_favorite"]
    e = ensure_numeric_cols(read_events_lf("user_category", start, end), event_cols)
    e_decay = time_decay_expr("event_date", end)
    e_score = ((pl.col("cnt_page_view") * w["page_view"] + pl.col("cnt_view_description") * w["view_description"] + pl.col("cnt_review_view") * w["review_view"] + pl.col("cnt_favorite") * w["favorite"] + pl.col("cnt_to_cart") * w["to_cart"]) * e_decay).cast(pl.Float32)
    e = e.with_columns(e_score.alias("pref_score")).group_by(["user_id", "catalogid"]).agg(pl.sum("pref_score").alias("pref_score"))
    o = ensure_numeric_cols(read_orders_lf("user_category", start, end), ["cnt_orders"])
    o_decay = time_decay_expr("created_date", end)
    o = o.with_columns((pl.col("cnt_orders") * w["order"] * o_decay).cast(pl.Float32).alias("pref_score")).group_by(["user_id", "catalogid"]).agg(pl.sum("pref_score").alias("pref_score"))
    pref = (
        pl.concat([e, o], how="vertical_relaxed")
        .group_by(["user_id", "catalogid"])
        .agg(pl.sum("pref_score").alias("user_catalog_pref_score"))
        .filter(pl.col("user_catalog_pref_score") > 0)
        .sort(["user_id", "user_catalog_pref_score", "catalogid"], descending=[False, True, False])
        .with_columns(pl.col("catalogid").cum_count().over("user_id").cast(pl.Int32).alias("user_catalog_pref_rank"))
        .filter(pl.col("user_catalog_pref_rank") <= int(CFG.user_category_top_categories))
    )
    pref = pref.join(item_catalog_lf_enriched().select("catalogid", "cat_l0").unique(), on="catalogid", how="left")
    pref = pref.with_columns(pl.col("cat_l0").fill_null(pl.col("catalogid")).cast(pl.Int64))
    df = collect_streaming(pref)
    df.write_parquet(out, compression="zstd")
    print(f"user category prefs: {df.height:,} -> {rel(out)}")
    return out


def build_category_item_dict(pop_table: pd.DataFrame) -> dict[int, tuple[np.ndarray, np.ndarray, np.ndarray]]:
    catalog = collect_streaming(item_catalog_lf_enriched().select("item_id", "catalogid")).to_pandas()
    pdf = pop_table[["item_id", "score", "global_rank"]].merge(catalog, on="item_id", how="left").dropna(subset=["catalogid"])
    pdf["catalogid"] = pdf["catalogid"].astype(np.int64)
    result = {}
    for cid, g in pdf.sort_values(["catalogid", "score", "item_id"], ascending=[True, False, True]).groupby("catalogid", sort=False):
        top = g.head(CFG.user_category_items_per_category)
        result[int(cid)] = (top["item_id"].astype(np.int64).to_numpy(), top["score"].astype(np.float32).to_numpy(), top["global_rank"].astype(np.int32).to_numpy())
    return result


def load_user_category_pref_dict(pref_path: Path) -> dict[int, list[tuple[int, float, int]]]:
    df = pd.read_parquet(pref_path, columns=["user_id", "catalogid", "user_catalog_pref_score", "user_catalog_pref_rank"])
    result: dict[int, list[tuple[int, float, int]]] = defaultdict(list)
    for row in df.itertuples(index=False):
        result[int(row.user_id)].append((int(row.catalogid), float(row.user_catalog_pref_score), int(row.user_catalog_pref_rank)))
    return dict(result)


def user_category_popular_part_for_users(user_ids: list[int], pref_by_user: dict[int, list[tuple[int, float, int]]], cat_items: dict[int, tuple[np.ndarray, np.ndarray, np.ndarray]], *, ordered_by_user: dict[int, set[int]], k: int) -> pl.DataFrame:
    up, ip, sp, rp = [], [], [], []
    for user_id in user_ids:
        prefs = pref_by_user.get(int(user_id), [])
        if not prefs:
            continue
        scores: dict[int, float] = {}
        banned = ordered_by_user.get(int(user_id), set()) if CFG.filter_ordered_items else set()
        for cid, pref_score, pref_rank in prefs:
            data = cat_items.get(int(cid))
            if data is None:
                continue
            items, item_scores, _ = data
            for it, s in zip(items, item_scores):
                it = int(it)
                if it in banned:
                    continue
                score = math.log1p(float(s)) * math.log1p(float(pref_score)) / max(1, int(pref_rank))
                if score > scores.get(it, -1.0):
                    scores[it] = score
        if not scores:
            continue
        items = np.fromiter(scores.keys(), dtype=np.int64)
        vals = np.fromiter(scores.values(), dtype=np.float32)
        kk = min(k, len(vals))
        top = np.argpartition(-vals, kk - 1)[:kk]
        top = top[np.argsort(-vals[top])]
        n = len(top)
        up.append(np.full(n, int(user_id), dtype=np.int64)); ip.append(items[top]); sp.append(vals[top]); rp.append(np.arange(1, n + 1, dtype=np.int32))
    if not up:
        return pl.DataFrame(schema={"user_id": pl.Int64, "item_id": pl.Int64, "source": pl.Utf8, "score": pl.Float32, "rank": pl.Int32, "in_user_category_popular": pl.Int8})
    return pl.DataFrame({"user_id": np.concatenate(up), "item_id": np.concatenate(ip), "score": np.concatenate(sp), "rank": np.concatenate(rp)}).with_columns(pl.lit("user_category_popular").alias("source"), pl.lit(1, dtype=pl.Int8).alias("in_user_category_popular"))


def generate_user_category_popular_candidates_to_dir(user_ids: list[int], pref_path: Path, pop_table: pd.DataFrame, out_dir: Path, *, ordered_by_user: dict[int, set[int]], k: int, n_shards: int = CFG.n_shards) -> Path:
    clean_output_dir(out_dir)
    pref_by_user = load_user_category_pref_dict(pref_path)
    cat_items = build_category_item_dict(pop_table)
    by_shard: dict[int, list[int]] = defaultdict(list)
    for u in user_ids:
        by_shard[int(u) % n_shards].append(int(u))
    rows = 0
    for shard_id in trange(n_shards, desc="UserCategoryPopular shards"):
        users = by_shard.get(shard_id, [])
        if not users:
            continue
        part = user_category_popular_part_for_users(users, pref_by_user, cat_items, ordered_by_user=ordered_by_user, k=k)
        if not part.is_empty():
            rows += write_single_shard(part, out_dir, shard_id, "user_category_popular.parquet")
        del part
        gc.collect()
    print(f"UserCategoryPopular candidates saved: rows={rows:,} -> {rel(out_dir)}")
    return out_dir

## CLIP/fCLIP profile candidates: HNSW + FAISS backends

In [9]:
# Rationale:
# - HNSW is the default CPU backend for production-like approximate retrieval.
# - FAISS is kept as an optional challenger/benchmark and is especially useful if GPU is available.
# Both sources are written separately (`clip_hnsw`, `clip_faiss`) so their contribution can be compared.


def raw_items_data_path() -> Path | None:
    candidates = [RAW_DIR / "ml_ozon_recsys_train_final_apparel_items_data", DATA_DIR / "ml_ozon_recsys_train_final_apparel_items_data"]
    return next((p for p in candidates if p.exists()), None)


def iter_parquet_files(path: Path) -> list[Path]:
    if path.is_file():
        return [path]
    return sorted(path.rglob("*.parquet"))


@dataclass
class ClipIndexBundle:
    backend: str
    index: object
    itemid_to_ann: dict[int, int]
    ann_to_itemid: np.ndarray
    embeddings: np.ndarray
    dim: int

    def query(self, vector: np.ndarray, n: int) -> np.ndarray:
        n = int(min(n, len(self.ann_to_itemid)))
        if n <= 0:
            return np.empty(0, dtype=np.int64)
        q = np.asarray(vector, dtype=np.float32).reshape(1, -1)
        if self.backend == "hnsw":
            labels, _ = self.index.knn_query(q, k=n)
            return np.asarray(labels[0], dtype=np.int64)
        if self.backend == "faiss":
            _, labels = self.index.search(q, n)
            labels = np.asarray(labels[0], dtype=np.int64)
            return labels[labels >= 0]
        raise ValueError(f"Unknown ANN backend: {self.backend}")


def build_or_load_clip_embeddings(item_map_path: Path, out_dir: Path, label: str) -> tuple[np.ndarray, np.ndarray, dict[int, int], int] | None:
    """Build/load normalized item fclip embedding matrix once per split.

    Returns:
        embeddings: [n_items_with_embedding, dim] float32, normalized if configured
        ann_to_itemid: internal ANN row -> original item_id
        itemid_to_ann: original item_id -> internal ANN row
        dim: embedding dimension
    """
    items_path = raw_items_data_path()
    if items_path is None:
        warnings.warn("Raw items data with fclip_embed not found; CLIP sources will be skipped.")
        return None

    ensure_dir(out_dir)
    emb_path = out_dir / f"clip_embeddings_{label}.npy"
    map_path = out_dir / f"clip_embeddings_{label}_mapping.parquet"
    meta_path = out_dir / f"clip_embeddings_{label}_meta.json"

    if emb_path.exists() and map_path.exists() and meta_path.exists() and not CFG.rebuild_aux:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        emb = np.load(emb_path, mmap_mode="r")
        mapping = pd.read_parquet(map_path).sort_values("ann_idx")
        ann_to_itemid = mapping["item_id"].astype(np.int64).to_numpy()
        itemid_to_ann = dict(zip(ann_to_itemid.astype(int), mapping["ann_idx"].astype(int)))
        return emb, ann_to_itemid, itemid_to_ann, int(meta["dim"])

    target_items = set(pd.read_parquet(item_map_path, columns=["item_id"])["item_id"].astype(int).tolist())
    vectors: list[np.ndarray] = []
    item_ids: list[int] = []
    dim = None

    for f in tqdm(iter_parquet_files(items_path), desc="Load fclip embeddings"):
        try:
            df = pl.read_parquet(f, columns=["item_id", "fclip_embed"])
        except Exception:
            continue
        if df.is_empty() or "fclip_embed" not in df.columns:
            continue
        df = df.filter(pl.col("item_id").is_in(target_items))
        if df.is_empty():
            continue
        for item_id, emb in zip(df["item_id"].to_list(), df["fclip_embed"].to_list()):
            if emb is None:
                continue
            vec = np.asarray(emb, dtype=np.float32)
            if vec.ndim != 1 or vec.size == 0 or not np.isfinite(vec).all():
                continue
            if CFG.normalize_clip_vectors:
                norm = float(np.linalg.norm(vec))
                if norm <= 0:
                    continue
                vec = vec / norm
            if dim is None:
                dim = int(vec.size)
            if int(vec.size) != int(dim):
                continue
            vectors.append(vec.astype(np.float32, copy=False))
            item_ids.append(int(item_id))
        del df
        gc.collect()

    if not vectors or dim is None:
        warnings.warn("No fclip embeddings were loaded; CLIP sources will be skipped.")
        return None

    embeddings = np.vstack(vectors).astype(np.float32, copy=False)
    ann_to_itemid = np.asarray(item_ids, dtype=np.int64)
    ann_idx = np.arange(len(ann_to_itemid), dtype=np.int32)
    itemid_to_ann = dict(zip(ann_to_itemid.astype(int), ann_idx.astype(int)))

    np.save(emb_path, embeddings)
    pl.DataFrame({"ann_idx": ann_idx, "item_id": ann_to_itemid}).write_parquet(map_path, compression="zstd")
    meta_path.write_text(json.dumps({"dim": dim, "items": len(ann_to_itemid), "built_at_utc": datetime.now(timezone.utc).isoformat()}, indent=2), encoding="utf-8")
    print(f"CLIP embeddings: items={len(ann_to_itemid):,}, dim={dim} -> {rel(emb_path)}")
    return embeddings, ann_to_itemid, itemid_to_ann, int(dim)


def build_or_load_clip_hnsw_index(item_map_path: Path, out_dir: Path, label: str) -> ClipIndexBundle | None:
    if not HAS_HNSWLIB:
        warnings.warn(f"hnswlib не установлен; clip_hnsw source будет пропущен. Ошибка импорта: {HNSWLIB_IMPORT_ERROR}")
        return None
    data = build_or_load_clip_embeddings(item_map_path, out_dir, label)
    if data is None:
        return None
    embeddings, ann_to_itemid, itemid_to_ann, dim = data
    index_path = out_dir / f"clip_hnsw_{label}.bin"
    meta_path = out_dir / f"clip_hnsw_{label}_meta.json"
    n = len(ann_to_itemid)

    idx = hnswlib.Index(space="cosine", dim=int(dim))
    if index_path.exists() and meta_path.exists() and not CFG.rebuild_aux:
        idx.load_index(str(index_path), max_elements=n)
        idx.set_ef(int(CFG.hnsw_ef_search))
        print(f"CLIP HNSW index loaded: items={n:,}, dim={dim} -> {rel(index_path)}")
        return ClipIndexBundle("hnsw", idx, itemid_to_ann, ann_to_itemid, embeddings, int(dim))

    idx.init_index(max_elements=n, ef_construction=int(CFG.hnsw_ef_construction), M=int(CFG.hnsw_m))
    idx.add_items(np.asarray(embeddings, dtype=np.float32), np.arange(n, dtype=np.int64), num_threads=int(CFG.hnsw_num_threads))
    idx.set_ef(int(CFG.hnsw_ef_search))
    idx.save_index(str(index_path))
    meta_path.write_text(json.dumps({"dim": dim, "items": n, "M": CFG.hnsw_m, "ef_construction": CFG.hnsw_ef_construction, "ef_search": CFG.hnsw_ef_search, "built_at_utc": datetime.now(timezone.utc).isoformat()}, indent=2), encoding="utf-8")
    print(f"CLIP HNSW index built: items={n:,}, dim={dim} -> {rel(index_path)}")
    return ClipIndexBundle("hnsw", idx, itemid_to_ann, ann_to_itemid, embeddings, int(dim))


def _maybe_gpu_faiss_index(index):
    if not CFG.faiss_use_gpu:
        return index
    try:
        res = faiss.StandardGpuResources()
        return faiss.index_cpu_to_gpu(res, 0, index)
    except Exception as e:
        warnings.warn(f"FAISS GPU requested but unavailable; using CPU. {type(e).__name__}: {e}")
        return index


def build_or_load_clip_faiss_index(item_map_path: Path, out_dir: Path, label: str) -> ClipIndexBundle | None:
    if not HAS_FAISS:
        warnings.warn(f"faiss не установлен; clip_faiss source будет пропущен. Ошибка импорта: {FAISS_IMPORT_ERROR}")
        return None
    data = build_or_load_clip_embeddings(item_map_path, out_dir, label)
    if data is None:
        return None
    embeddings, ann_to_itemid, itemid_to_ann, dim = data
    index_path = out_dir / f"clip_faiss_{CFG.faiss_index_type}_{label}.index"
    meta_path = out_dir / f"clip_faiss_{CFG.faiss_index_type}_{label}_meta.json"
    n = len(ann_to_itemid)

    if index_path.exists() and meta_path.exists() and not CFG.rebuild_aux:
        idx = faiss.read_index(str(index_path))
        idx = _maybe_gpu_faiss_index(idx)
        print(f"CLIP FAISS index loaded: items={n:,}, dim={dim}, type={CFG.faiss_index_type} -> {rel(index_path)}")
        return ClipIndexBundle("faiss", idx, itemid_to_ann, ann_to_itemid, embeddings, int(dim))

    x = np.asarray(embeddings, dtype=np.float32)
    if CFG.faiss_index_type == "flat" or n < max(10_000, int(CFG.faiss_nlist) * 20):
        idx = faiss.IndexFlatIP(int(dim))
    elif CFG.faiss_index_type == "ivf_flat":
        nlist = int(min(CFG.faiss_nlist, max(1, math.sqrt(n) * 4)))
        quantizer = faiss.IndexFlatIP(int(dim))
        idx = faiss.IndexIVFFlat(quantizer, int(dim), nlist, faiss.METRIC_INNER_PRODUCT)
        sample_n = min(n, int(CFG.faiss_train_sample_size))
        rng = np.random.default_rng(42)
        train_idx = rng.choice(n, size=sample_n, replace=False) if sample_n < n else np.arange(n)
        idx.train(np.ascontiguousarray(x[train_idx]))
        idx.nprobe = int(min(CFG.faiss_nprobe, nlist))
    else:
        raise ValueError(f"Unsupported CFG.faiss_index_type={CFG.faiss_index_type!r}")

    idx.add(x)
    faiss.write_index(faiss.index_gpu_to_cpu(idx) if hasattr(faiss, "index_gpu_to_cpu") and "Gpu" in type(idx).__name__ else idx, str(index_path))
    idx = _maybe_gpu_faiss_index(idx)
    if hasattr(idx, "nprobe"):
        idx.nprobe = int(min(CFG.faiss_nprobe, getattr(idx, "nlist", CFG.faiss_nprobe)))
    meta_path.write_text(json.dumps({"dim": dim, "items": n, "index_type": CFG.faiss_index_type, "nlist": getattr(idx, "nlist", None), "nprobe": getattr(idx, "nprobe", None), "built_at_utc": datetime.now(timezone.utc).isoformat()}, indent=2), encoding="utf-8")
    print(f"CLIP FAISS index built: items={n:,}, dim={dim}, type={CFG.faiss_index_type} -> {rel(index_path)}")
    return ClipIndexBundle("faiss", idx, itemid_to_ann, ann_to_itemid, embeddings, int(dim))


def build_user_clip_profile(row, item_ids_by_idx: np.ndarray, bundle: ClipIndexBundle) -> np.ndarray | None:
    if row is None or row.nnz == 0:
        return None
    idxs = row.indices
    vals = row.data.astype(np.float32, copy=False)
    if len(vals) > CFG.clip_history_k:
        top = np.argpartition(-vals, CFG.clip_history_k - 1)[:CFG.clip_history_k]
        top = top[np.argsort(-vals[top])]
    else:
        top = np.argsort(-vals)
    vec = None
    denom = 0.0
    for pos in top:
        item_id = int(item_ids_by_idx[int(idxs[pos])])
        ann_ix = bundle.itemid_to_ann.get(item_id)
        if ann_ix is None:
            continue
        v = np.asarray(bundle.embeddings[int(ann_ix)], dtype=np.float32)
        w = float(vals[pos])
        vec = v * w if vec is None else vec + v * w
        denom += w
    if vec is None or denom <= 0:
        return None
    norm = float(np.linalg.norm(vec))
    if norm <= 0:
        return None
    return (vec / norm).astype(np.float32)


def clip_profile_part_for_user_idxs(bundle: ClipIndexBundle, matrix: csr_matrix, user_idxs: list[int], user_ids_by_idx: np.ndarray, item_ids_by_idx: np.ndarray, ordered_by_user_idx: dict[int, set[int]], *, k: int, source_name: str) -> pl.DataFrame:
    """Generate CLIP candidates for one shard using batched ANN queries.

    v4 queried HNSW/FAISS once per user, which is very slow for 300k+ users.
    This version still builds user profiles in Python, but sends all profiles from
    the shard to the ANN backend in one batch. On CPU HNSW this is usually the
    difference between hours and minutes.
    """
    up, ip, sp, rp = [], [], [], []
    flag_col = f"in_{source_name}"

    kept_user_idxs: list[int] = []
    profiles: list[np.ndarray] = []
    for u_idx in tqdm(user_idxs, leave=False, desc=f"{source_name} profiles"):
        row = matrix.getrow(int(u_idx))
        prof = build_user_clip_profile(row, item_ids_by_idx, bundle)
        if prof is None:
            continue
        kept_user_idxs.append(int(u_idx))
        profiles.append(prof)

    if not profiles:
        return pl.DataFrame(schema={"user_id": pl.Int64, "item_id": pl.Int64, "source": pl.Utf8, "score": pl.Float32, "rank": pl.Int32, flag_col: pl.Int8})

    q = np.ascontiguousarray(np.vstack(profiles).astype(np.float32, copy=False))
    n_query = int(min(CFG.clip_oversample, len(bundle.ann_to_itemid)))

    try:
        if bundle.backend == "hnsw":
            ann_labels, _ = bundle.index.knn_query(q, k=n_query)
        elif bundle.backend == "faiss":
            _, ann_labels = bundle.index.search(q, n_query)
        else:
            raise ValueError(f"Unknown ANN backend: {bundle.backend}")
    except Exception as e:
        warnings.warn(f"{source_name} batched query failed; source shard will be empty. {type(e).__name__}: {e}")
        return pl.DataFrame(schema={"user_id": pl.Int64, "item_id": pl.Int64, "source": pl.Utf8, "score": pl.Float32, "rank": pl.Int32, flag_col: pl.Int8})

    ann_labels = np.asarray(ann_labels, dtype=np.int64)
    emb = np.asarray(bundle.embeddings, dtype=np.float32)

    for row_no, u_idx in enumerate(kept_user_idxs):
        prof = q[row_no]
        banned_item_ids = set()
        if CFG.filter_ordered_items:
            for bi in ordered_by_user_idx.get(int(u_idx), set()):
                if 0 <= int(bi) < len(item_ids_by_idx):
                    banned_item_ids.add(int(item_ids_by_idx[int(bi)]))
        items, scores = [], []
        for ann_ix in ann_labels[row_no]:
            ann_ix = int(ann_ix)
            if ann_ix < 0 or ann_ix >= len(bundle.ann_to_itemid):
                continue
            item_id = int(bundle.ann_to_itemid[ann_ix])
            if item_id in banned_item_ids:
                continue
            score = float(np.dot(prof, emb[ann_ix]))
            items.append(item_id)
            scores.append(score)
            if len(items) >= k:
                break
        if not items:
            continue
        n = len(items)
        up.append(np.full(n, user_ids_by_idx[int(u_idx)], dtype=np.int64))
        ip.append(np.asarray(items, dtype=np.int64))
        sp.append(np.asarray(scores, dtype=np.float32))
        rp.append(np.arange(1, n + 1, dtype=np.int32))

    if not up:
        return pl.DataFrame(schema={"user_id": pl.Int64, "item_id": pl.Int64, "source": pl.Utf8, "score": pl.Float32, "rank": pl.Int32, flag_col: pl.Int8})
    return pl.DataFrame({"user_id": np.concatenate(up), "item_id": np.concatenate(ip), "score": np.concatenate(sp), "rank": np.concatenate(rp)}).with_columns(pl.lit(source_name).alias("source"), pl.lit(1, dtype=pl.Int8).alias(flag_col))

def generate_clip_profile_candidates_to_dir(bundle: ClipIndexBundle, matrix: csr_matrix, user_idxs: list[int], user_map_path: Path, item_map_path: Path, ordered_by_user_idx: dict[int, set[int]], out_dir: Path, *, k: int, source_name: str, n_shards: int = CFG.n_shards) -> Path:
    clean_output_dir(out_dir)
    user_ids_by_idx, item_ids_by_idx = mapping_arrays(user_map_path, item_map_path)
    by_shard: dict[int, list[int]] = defaultdict(list)
    for u_idx in user_idxs:
        by_shard[int(user_ids_by_idx[int(u_idx)]) % n_shards].append(int(u_idx))
    rows = 0
    for shard_id in trange(n_shards, desc=f"{source_name} candidate shards"):
        idxs = by_shard.get(shard_id, [])
        if not idxs:
            continue
        part = clip_profile_part_for_user_idxs(bundle, matrix, idxs, user_ids_by_idx, item_ids_by_idx, ordered_by_user_idx, k=k, source_name=source_name)
        if not part.is_empty():
            rows += write_single_shard(part, out_dir, shard_id, f"{source_name}.parquet")
        del part
        gc.collect()
    print(f"{source_name} candidates saved: rows={rows:,} -> {rel(out_dir)}")
    return out_dir

## Merge sources and add light candidate features

In [10]:
SOURCE_DEFS = {
    "top_popular": {"score_col": "top_popular_score", "rank_col": "top_popular_rank", "flag_col": "in_top_popular"},
    "als": {"score_col": "als_score", "rank_col": "als_rank", "flag_col": "in_als"},
    "i2i_bm25": {"score_col": "i2i_bm25_score", "rank_col": "i2i_bm25_rank", "flag_col": "in_i2i_bm25"},
    "clip_hnsw": {"score_col": "clip_hnsw_score", "rank_col": "clip_hnsw_rank", "flag_col": "in_clip_hnsw"},
    "clip_faiss": {"score_col": "clip_faiss_score", "rank_col": "clip_faiss_rank", "flag_col": "in_clip_faiss"},
    "user_category_popular": {"score_col": "user_category_popular_score", "rank_col": "user_category_popular_rank", "flag_col": "in_user_category_popular"},
}


def _select_source_lf(lf: pl.LazyFrame, source_name: str) -> pl.LazyFrame:
    defs = SOURCE_DEFS[source_name]
    exprs = [pl.col("user_id").cast(pl.Int64), pl.col("item_id").cast(pl.Int64)]
    for name, d in SOURCE_DEFS.items():
        if name == source_name:
            exprs.extend([
                pl.col("score").cast(pl.Float32).alias(d["score_col"]),
                pl.col("rank").cast(pl.Int32).alias(d["rank_col"]),
                pl.lit(1, dtype=pl.Int8).alias(d["flag_col"]),
            ])
        else:
            exprs.extend([
                pl.lit(None, dtype=pl.Float32).alias(d["score_col"]),
                pl.lit(None, dtype=pl.Int32).alias(d["rank_col"]),
                pl.lit(0, dtype=pl.Int8).alias(d["flag_col"]),
            ])
    # Extra stable popularity feature.
    if source_name == "top_popular" and "global_pop_rank" in lazy_columns(lf):
        exprs.append(pl.col("global_pop_rank").cast(pl.Int32))
    else:
        exprs.append(pl.lit(None, dtype=pl.Int32).alias("global_pop_rank"))
    return lf.select(exprs)


def merge_candidate_dirs_by_shard(source_dirs: dict[str, Path], out_dir: Path, *, pref_path: Path | None, n_shards: int = CFG.n_shards) -> Path:
    clean_output_dir(out_dir)
    total_rows = 0
    catalog_lf = item_catalog_lf_enriched().select("item_id", "catalogid", "cat_l0").unique()
    pref_catalog_lf = None
    pref_l0_lf = None
    if pref_path is not None and pref_path.exists():
        pref_catalog_lf = scan_parquet(pref_path, hive_partitioning=False).select(
            pl.col("user_id").cast(pl.Int64),
            pl.col("catalogid").cast(pl.Int64),
            pl.col("user_catalog_pref_score").cast(pl.Float32),
            pl.col("user_catalog_pref_rank").cast(pl.Int32),
        )
        pref_l0_lf = (
            scan_parquet(pref_path, hive_partitioning=False)
            .select(pl.col("user_id").cast(pl.Int64), pl.col("cat_l0").cast(pl.Int64), pl.col("user_catalog_pref_score").cast(pl.Float32))
            .group_by(["user_id", "cat_l0"])
            .agg(pl.max("user_catalog_pref_score").alias("user_cat_l0_pref_score"))
            .sort(["user_id", "user_cat_l0_pref_score", "cat_l0"], descending=[False, True, False])
            .with_columns(pl.col("cat_l0").cum_count().over("user_id").cast(pl.Int32).alias("user_cat_l0_pref_rank"))
        )

    for shard_id in trange(n_shards, desc="Merge enhanced candidate shards"):
        parts = []
        for source_name, source_dir in source_dirs.items():
            lf = scan_candidate_shard_or_none(source_dir, shard_id)
            if lf is not None:
                parts.append(_select_source_lf(lf, source_name))
        if not parts:
            continue
        agg_exprs = []
        for name, d in SOURCE_DEFS.items():
            agg_exprs.extend([
                pl.max(d["score_col"]).alias(d["score_col"]),
                pl.min(d["rank_col"]).alias(d["rank_col"]),
                pl.max(d["flag_col"]).alias(d["flag_col"]),
            ])
        agg_exprs.append(pl.min("global_pop_rank").alias("global_pop_rank"))
        merged = pl.concat(parts, how="vertical_relaxed").group_by(["user_id", "item_id"]).agg(*agg_exprs)
        flag_cols = [d["flag_col"] for d in SOURCE_DEFS.values()]
        rank_cols = [d["rank_col"] for d in SOURCE_DEFS.values()]
        score_cols = [d["score_col"] for d in SOURCE_DEFS.values()]
        merged = merged.with_columns(
            pl.sum_horizontal(*[pl.col(c).fill_null(0).cast(pl.Int16) for c in flag_cols]).cast(pl.Int8).alias("sources_count"),
            pl.min_horizontal(*[pl.col(c).fill_null(1_000_000_000) for c in rank_cols]).cast(pl.Int32).alias("min_source_rank"),
            pl.max_horizontal(*[pl.col(c).fill_null(0.0) for c in score_cols]).cast(pl.Float32).alias("max_source_score"),
        )
        merged = merged.join(catalog_lf, on="item_id", how="left").with_columns(
            pl.col("catalogid").fill_null(-1).cast(pl.Int64),
            pl.col("cat_l0").fill_null(-1).cast(pl.Int64),
        )
        if pref_catalog_lf is not None:
            merged = merged.join(pref_catalog_lf, on=["user_id", "catalogid"], how="left")
        else:
            merged = merged.with_columns(pl.lit(None, dtype=pl.Float32).alias("user_catalog_pref_score"), pl.lit(None, dtype=pl.Int32).alias("user_catalog_pref_rank"))
        if pref_l0_lf is not None:
            merged = merged.join(pref_l0_lf, on=["user_id", "cat_l0"], how="left")
        else:
            merged = merged.with_columns(pl.lit(None, dtype=pl.Float32).alias("user_cat_l0_pref_score"), pl.lit(None, dtype=pl.Int32).alias("user_cat_l0_pref_rank"))
        merged = merged.with_columns(
            pl.col("user_catalog_pref_score").is_not_null().cast(pl.Int8).alias("same_catalogid"),
            pl.col("user_cat_l0_pref_score").is_not_null().cast(pl.Int8).alias("same_cat_l0"),
        )
        df = collect_streaming(merged)
        if df.is_empty():
            continue
        df = (
            df.sort(["user_id", "sources_count", "min_source_rank", "max_source_score", "item_id"], descending=[False, True, False, True, False])
            .with_columns(pl.col("item_id").cum_count().over("user_id").cast(pl.Int32).alias("candidate_rank"))
            .filter(pl.col("candidate_rank") <= int(CFG.merged_max_candidates_per_user))
        )
        total_rows += write_single_shard(df, out_dir, shard_id, "merged.parquet")
        del df, merged
        gc.collect()
    print(f"Merged enhanced candidates saved: rows={total_rows:,} -> {rel(out_dir)}")
    return out_dir



def candidate_dir_looks_complete(path: Path, *, n_shards: int = CFG.n_shards) -> bool:
    """Return True only when a sharded candidate directory appears complete.

    This avoids reusing a partially written source after an interrupted notebook run.
    The project candidate writers create one parquet file per user_shard for these full-user splits.
    """
    if not path.exists():
        return False
    files = candidate_files(path)
    if not files:
        return False
    non_empty_files = [p for p in files if p.stat().st_size > 0]
    if len(non_empty_files) < n_shards:
        return False
    existing_shards = {
        int(p.parent.name.split("=", 1)[1])
        for p in non_empty_files
        if p.parent.name.startswith("user_shard=") and p.parent.name.split("=", 1)[1].isdigit()
    }
    return len(existing_shards) >= n_shards


def should_rebuild_source(source_name: str) -> bool:
    if CFG.rebuild_candidate_dirs:
        return True
    if source_name == "i2i_bm25" and CFG.rebuild_i2i_bm25_candidates:
        return True
    return False


def _use_or_build_source(split_name: str, source_name: str, out_dir: Path, build_fn) -> Path:
    if candidate_dir_looks_complete(out_dir) and not should_rebuild_source(source_name):
        print(f"{split_name}/{source_name}: source reused -> {rel(out_dir)}")
        return out_dir
    if path_has_parquet(out_dir):
        print(f"{split_name}/{source_name}: source exists but looks incomplete; rebuilding -> {rel(out_dir)}")
    return build_fn()


def build_split_candidates(split_name: str, *, context_start: date, context_end: date, user_ids: list[int]) -> Path:
    split_artifacts_dir = PROCESSED_DIR / split_name
    ensure_dir(split_artifacts_dir)
    merged_out = CANDIDATES_DIR / "merged" / split_name
    if path_has_parquet(merged_out) and not CFG.rebuild_candidate_dirs and not CFG.rebuild_merged_candidates:
        print(f"{split_name}: merged candidates already exists -> {rel(merged_out)}")
        return merged_out

    source_dirs: dict[str, Path] = {}
    popular = build_top_popular_table(context_start, context_end, out_dir=split_artifacts_dir, label=split_name)
    ordered_by_user = build_ordered_by_user(context_start, context_end) if CFG.filter_ordered_items else {}
    seen_events_by_user = build_seen_events_by_user(context_start, context_end) if CFG.filter_seen_event_items else {}
    pref_path = build_user_category_preferences(context_start, context_end, split_artifacts_dir, split_name)

    if CFG.run_top_popular:
        d = CANDIDATES_DIR / "top_popular" / split_name
        source_dirs["top_popular"] = _use_or_build_source(
            split_name, "top_popular", d,
            lambda d=d: generate_top_popular_candidates_to_dir(user_ids, popular, d, ordered_by_user=ordered_by_user, seen_events_by_user=seen_events_by_user, k=CFG.top_popular_k),
        )

    matrix = user_map_path = item_map_path = user_idxs = ordered_by_user_idx = None
    if CFG.run_als or CFG.run_i2i_bm25 or CFG.run_clip_hnsw or CFG.run_clip_faiss:
        csr_dir = split_artifacts_dir / "csr"
        matrix, user_map_path, item_map_path = build_csr_from_datamart(context_start, context_end, csr_dir, n_shards=CFG.csr_n_shards)
        user_idxs = make_user_idx_list(user_ids, user_map_path)
        print(f"{split_name}: warm users in CSR: {len(user_idxs):,} / {len(user_ids):,}")
        ordered_by_user_idx = build_ordered_by_user(context_start, context_end, as_idx=True, user_map_path=user_map_path, item_map_path=item_map_path) if CFG.filter_ordered_items else {}

    if CFG.run_als and matrix is not None:
        d = CANDIDATES_DIR / "als" / split_name
        if candidate_dir_looks_complete(d) and not should_rebuild_source("als"):
            print(f"{split_name}/als: source reused -> {rel(d)}")
            source_dirs["als"] = d
        else:
            model = train_or_load_als(matrix, MODELS_DIR / f"als_{split_name}.npz")
            source_dirs["als"] = generate_als_candidates_to_dir(model, matrix, user_idxs, user_map_path, item_map_path, ordered_by_user_idx, d, k=CFG.als_k)

    if CFG.run_i2i_bm25 and matrix is not None:
        d = CANDIDATES_DIR / "i2i_bm25" / split_name
        if candidate_dir_looks_complete(d) and not should_rebuild_source("i2i_bm25"):
            print(f"{split_name}/i2i_bm25: source reused -> {rel(d)}")
            source_dirs["i2i_bm25"] = d
        else:
            try:
                i2i_model = train_or_load_i2i_bm25(matrix, MODELS_DIR / f"i2i_bm25_scipy_{split_name}.pkl")
                source_dirs["i2i_bm25"] = generate_i2i_bm25_candidates_to_dir(i2i_model, matrix, user_idxs, user_map_path, item_map_path, ordered_by_user_idx, d, k=CFG.i2i_bm25_k)
            except Exception as e:
                warnings.warn(f"I2I BM25 failed for split={split_name}; source will be skipped. {type(e).__name__}: {e}")

    if CFG.run_clip_hnsw and matrix is not None:
        d = CANDIDATES_DIR / "clip_hnsw" / split_name
        if candidate_dir_looks_complete(d) and not should_rebuild_source("clip_hnsw"):
            print(f"{split_name}/clip_hnsw: source reused -> {rel(d)}")
            source_dirs["clip_hnsw"] = d
        else:
            bundle = build_or_load_clip_hnsw_index(item_map_path, split_artifacts_dir, split_name)
            if bundle is not None:
                source_dirs["clip_hnsw"] = generate_clip_profile_candidates_to_dir(bundle, matrix, user_idxs, user_map_path, item_map_path, ordered_by_user_idx, d, k=CFG.clip_hnsw_k, source_name="clip_hnsw")

    if CFG.run_clip_faiss and matrix is not None:
        d = CANDIDATES_DIR / "clip_faiss" / split_name
        if candidate_dir_looks_complete(d) and not should_rebuild_source("clip_faiss"):
            print(f"{split_name}/clip_faiss: source reused -> {rel(d)}")
            source_dirs["clip_faiss"] = d
        else:
            bundle = build_or_load_clip_faiss_index(item_map_path, split_artifacts_dir, split_name)
            if bundle is not None:
                source_dirs["clip_faiss"] = generate_clip_profile_candidates_to_dir(bundle, matrix, user_idxs, user_map_path, item_map_path, ordered_by_user_idx, d, k=CFG.clip_faiss_k, source_name="clip_faiss")

    if CFG.run_user_category_popular:
        d = CANDIDATES_DIR / "user_category_popular" / split_name
        source_dirs["user_category_popular"] = _use_or_build_source(
            split_name, "user_category_popular", d,
            lambda d=d: generate_user_category_popular_candidates_to_dir(user_ids, pref_path, popular, d, ordered_by_user=ordered_by_user, k=CFG.user_category_popular_k),
        )

    if not source_dirs:
        raise RuntimeError(f"{split_name}: no candidate sources were generated")
    return merge_candidate_dirs_by_shard(source_dirs, merged_out, pref_path=pref_path, n_shards=CFG.n_shards)

## Build ranker_train/val/test candidates

In [11]:
val_gt = build_ground_truth(CFG.val_target_start, CFG.val_target_end)
train_gt = build_ground_truth(CFG.ranker_train_target_start, CFG.ranker_train_target_end)
train_users = sorted(train_gt.keys())
val_users = sorted(val_gt.keys())
test_users = load_test_users_fallback(CFG.test_target_start, CFG.test_target_end)
print(f"ranker_train users with GT: {len(train_users):,}")
print(f"val users with GT:          {len(val_users):,}")
print(f"test target users:          {len(test_users):,}")


train_candidates_path = build_split_candidates(
    "ranker_train",
    context_start=CFG.ranker_train_context_start,
    context_end=CFG.ranker_train_context_end,
    user_ids=train_users,
)

val_candidates_path = build_split_candidates(
    "val",
    context_start=CFG.val_context_start,
    context_end=CFG.val_context_end,
    user_ids=val_users,
)

test_candidates_path = build_split_candidates(
    "test",
    context_start=CFG.full_context_start,
    context_end=CFG.full_context_end,
    user_ids=test_users,
)

print("train_candidates_path:", rel(train_candidates_path))
print("val_candidates_path:  ", rel(val_candidates_path))
print("test_candidates_path: ", rel(test_candidates_path))

Raw test orders not found. Fallback: users from datamart orders in test window.
ranker_train users with GT: 376,016
val users with GT:          359,928
test target users:          346,759
active items: 3,511,345 -> data\processed\candidate_generators_enhanced\ranker_train\active_items_days=14_min_deliv=0_2025-06-03.parquet
TopPopular table ranker_train: 3,474,573 items -> data\processed\candidate_generators_enhanced\ranker_train\top_popular_table_ranker_train.parquet
user category prefs: 4,450,665 -> data\processed\candidate_generators_enhanced\ranker_train\user_category_preferences_ranker_train.parquet


TopPopular shards:   0%|          | 0/128 [00:00<?, ?it/s]

TopPopular candidates saved: rows=75,203,200 -> data\datamart\candidates\top_popular\ranker_train
active items: 3,511,345 -> data\processed\candidate_generators_enhanced\ranker_train\csr\active_items_days=14_min_deliv=0_2025-06-03.parquet
users: 942,997 -> data\processed\candidate_generators_enhanced\ranker_train\csr\user_mapping.parquet
items: 3,511,345 -> data\processed\candidate_generators_enhanced\ranker_train\csr\item_mapping.parquet


Build CSR shards:   0%|          | 0/32 [00:00<?, ?it/s]

CSR: shape=(942997, 3511345), nnz=213,061,987 -> data\processed\candidate_generators_enhanced\ranker_train\csr\interaction_matrix.npz
ranker_train: warm users in CSR: 374,109 / 376,016
ALS loaded: models\candidate_generators_enhanced\als_ranker_train.npz


ALS candidate shards:   0%|          | 0/128 [00:00<?, ?it/s]

... (807 outputs left out)

UserCategoryPopular shards:   0%|          | 0/128 [00:00<?, ?it/s]

UserCategoryPopular candidates saved: rows=69,051,044 -> data\datamart\candidates\user_category_popular\test


Merge enhanced candidate shards:   0%|          | 0/128 [00:00<?, ?it/s]

Merged enhanced candidates saved: rows=242,202,929 -> data\datamart\candidates\merged\test
train_candidates_path: data\datamart\candidates\merged\ranker_train
val_candidates_path:   data\datamart\candidates\merged\val
test_candidates_path:  data\datamart\candidates\merged\test


## Reports and checks

In [12]:
val_metrics = evaluate_candidate_dir(val_candidates_path, val_gt, rank_col="candidate_rank", label="enhanced_candidates")
source_contribution_val = source_contribution_recall_report(val_candidates_path, val_gt, rank_col="candidate_rank")

def check_candidate_dir(path: Path, name: str, *, check_duplicates: bool = False) -> pd.DataFrame:
    files = candidate_files(path)
    if not files:
        raise AssertionError(f"{name}: no parquet files in {path}")
    rows = users = dups = 0
    for shard_id in trange(CFG.n_shards, desc=f"Check {name}"):
        lf = scan_candidate_shard_or_none(path, shard_id)
        if lf is None:
            continue
        stats = collect_streaming(lf.select(pl.len().alias("rows"), pl.col("user_id").n_unique().alias("users"))).to_dicts()[0]
        rows += int(stats["rows"]); users += int(stats["users"])
        if check_duplicates:
            dup = collect_streaming(lf.group_by(["user_id", "item_id"]).len().filter(pl.col("len") > 1).select(pl.len().alias("n"))).to_dicts()[0]["n"]
            dups += int(dup)
    out = pd.DataFrame([{"name": name, "path": rel(path), "files": len(files), "rows": rows, "users_by_shard_sum": users, "duplicated_pairs": dups if check_duplicates else np.nan}])
    if check_duplicates and dups > 0:
        raise AssertionError(f"{name}: duplicated user-item rows")
    return out

candidate_dir_checks = pd.concat([
    check_candidate_dir(train_candidates_path, "ranker_train", check_duplicates=True),
    check_candidate_dir(val_candidates_path, "val", check_duplicates=True),
    check_candidate_dir(test_candidates_path, "test", check_duplicates=True),
], ignore_index=True)


def write_parquet(df: pd.DataFrame, path: Path) -> None:
    ensure_dir(path.parent)
    df.to_parquet(path, index=False)
    print(f"saved: {rel(path)} rows={len(df):,}")

write_parquet(val_metrics, REPORTS_DIR / "candidate_generators_enhanced_val_metrics.parquet")
write_parquet(source_contribution_val, REPORTS_DIR / "candidate_generators_enhanced_val_source_contribution.parquet")
write_parquet(candidate_dir_checks, REPORTS_DIR / "candidate_generators_enhanced_dir_checks.parquet")

manifest = {
    "built_at_utc": datetime.now(timezone.utc).isoformat(),
    "config": {k: str(v) if isinstance(v, date) else v for k, v in asdict(CFG).items()},
    "sources": list(SOURCE_DEFS.keys()),
    "outputs": {
        "ranker_train_candidates": rel(train_candidates_path),
        "val_candidates": rel(val_candidates_path),
        "test_candidates": rel(test_candidates_path),
        "val_metrics": rel(REPORTS_DIR / "candidate_generators_enhanced_val_metrics.parquet"),
        "source_contribution_val": rel(REPORTS_DIR / "candidate_generators_enhanced_val_source_contribution.parquet"),
    },
    "notes": [
        "This notebook supersedes the old TopPopular+ALS-only candidate generator.",
        "The merged pool is wider than K=200 and is intended for reranking.",
        "ranker_train/val/test are candidate roles; features in the reranker are still point-in-time by as_of_date.",
        "CLIP HNSW source is skipped automatically if hnswlib or fclip embeddings are unavailable; install with `pip install hnswlib` to enable.",
        "CLIP FAISS source is optional and skipped if faiss is unavailable; install with `pip install faiss-cpu` or conda package to enable.",
        "I2I BM25 v6 uses a pure SciPy chunked BM25+cosine backend and does not call implicit BM25Recommender.",
    ],
}
manifest_path = CANDIDATES_DIR / "manifest_candidate_generators_enhanced.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"saved: {rel(manifest_path)}")

print("Validation metrics:")
display(val_metrics)
print("Source contribution:")
display(source_contribution_val)
print("Dir checks:")
display(candidate_dir_checks)

Evaluate enhanced_candidates:   0%|          | 0/128 [00:00<?, ?it/s]

Source contribution:   0%|          | 0/128 [00:00<?, ?it/s]

Check ranker_train:   0%|          | 0/128 [00:00<?, ?it/s]

Check val:   0%|          | 0/128 [00:00<?, ?it/s]

Check test:   0%|          | 0/128 [00:00<?, ?it/s]

saved: artifacts\reports\candidate_generators_enhanced_val_metrics.parquet rows=5
saved: artifacts\reports\candidate_generators_enhanced_val_source_contribution.parquet rows=5
saved: artifacts\reports\candidate_generators_enhanced_dir_checks.parquet rows=3
saved: data\datamart\candidates\manifest_candidate_generators_enhanced.json
Validation metrics:


,model,k,users_total_gt,users_evaluated,coverage_users,avg_candidates_per_user,p50_candidates_per_user,p95_candidates_per_user,users_with_less_than_k,unique_items,hit_rate,recall,ndcg,map,mrr
0,enhanced_candidates,20,359928,359928,1.0,698.320442,700.0,700.0,0,3426893,0.030059,0.012853,0.006312,0.003013,0.006905
1,enhanced_candidates,50,359928,359928,1.0,698.320442,700.0,700.0,0,3426893,0.061157,0.023976,0.009198,0.003376,0.007839
2,enhanced_candidates,100,359928,359928,1.0,698.320442,700.0,700.0,0,3426893,0.110186,0.042449,0.013222,0.003664,0.008524
3,enhanced_candidates,200,359928,359928,1.0,698.320442,700.0,700.0,0,3426893,0.164541,0.064736,0.017515,0.003857,0.008917
4,enhanced_candidates,700,359928,359928,1.0,698.320442,700.0,700.0,1251,3426893,0.262947,0.108768,0.024885,0.004020,0.009196


Source contribution:


,k,total_gt_items,found_union,found_union_recall_micro,in_als,in_als_recall_micro,in_clip_faiss,in_clip_faiss_recall_micro,in_clip_hnsw,in_clip_hnsw_recall_micro,in_i2i_bm25,in_i2i_bm25_recall_micro,in_top_popular,in_top_popular_recall_micro,in_user_category_popular,in_user_category_popular_recall_micro
0,20,1576713,11573,0.007340,9376,0.005947,0,0.0,4279,0.002714,4688,0.002973,4097,0.002598,3891,0.002468
1,50,1576713,24700,0.015666,16834,0.010677,0,0.0,6917,0.004387,12407,0.007869,5921,0.003755,5760,0.003653
2,100,1576713,47342,0.030026,23119,0.014663,0,0.0,10273,0.006515,26985,0.017115,7079,0.004490,6795,0.004310
3,200,1576713,76042,0.048228,29221,0.018533,0,0.0,13296,0.008433,45977,0.029160,7770,0.004928,7181,0.004554
4,700,1576713,140798,0.089298,45939,0.029136,0,0.0,17312,0.010980,86936,0.055137,9797,0.006214,8219,0.005213


Dir checks:


,name,path,files,rows,users_by_shard_sum,duplicated_pairs
0,ranker_train,data\datamart\candidates\merged\ranker_train,128,262241074,376016,0
1,val,data\datamart\candidates\merged\val,128,251345080,359928,0
2,test,data\datamart\candidates\merged\test,128,242202929,346759,0
